# MoE dispatch with expert capacity and weighted combine

After routing, a mixture-of-experts layer must **group** tokens by expert, run each expert once on its group (one dense matmul per expert, not one per token), and **scatter** the results back, weighted by the gates. Fixed-size kernels also impose an expert **capacity**: an expert accepts at most $C$ assignments per batch and the rest are dropped.

Write `moe_forward(x, idx, gates, W1, W2, capacity=None)` in PyTorch.

## Inputs (float64 unless stated)
- `x`: `(N, D)` tokens. `idx`: int64 `(N, k)`, the experts chosen for each token (distinct within a row). `gates`: `(N, k)`.
- `W1`: `(E, D, F)`, `W2`: `(E, F, D)`. Expert $e$ computes $f_e(u) = \mathrm{relu}(u\,W_1[e])\,W_2[e]$.
- `capacity`: int $C \ge 0$, or `None` for no limit.

## Capacity rule
Process assignments in **priority order**: slot 0 of tokens $0, 1, \dots, N-1$, then slot 1 of tokens $0, \dots, N-1$, and so on (every token's first choice beats anyone's second choice). Each expert keeps the first $C$ assignments it receives in that order and drops the rest.

## Output
A tuple `(y, kept)`:
- `y`: `(N, D)`, with $y_n = \sum_{s\,:\,\text{kept}[n,s]} \texttt{gates}[n,s]\; f_{\texttt{idx}[n,s]}(x_n)$. A token whose assignments were all dropped gets a zero row (the residual connection outside the layer carries it).
- `kept`: bool `(N, k)`.

Loop over experts if you like, but not over tokens.

This problem runs in **Colab** (PyTorch isn't available in the browser). The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch

def moe_forward(x, idx, gates, W1, W2, capacity=None):
    """Return (y (N,D), kept (N,k) bool)."""
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[1.0], [2.0], [-1.0]], "dtype": "float64"}, {"__nd__": [[0], [1], [0]], "dtype": "int64"}, {"__nd__": [[1.0], [1.0], [1.0]], "dtype": "float64"}, {"__nd__": [[[1.0]], [[2.0]]], "dtype": "float64"}, {"__nd__": [[[1.0]], [[1.0]]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[1.0], [4.0], [0.0]], "dtype": "float64"}, {"__nd__": [[true], [true], [true]], "dtype": "bool"}]}}, {"a": [{"__nd__": [[1.0], [2.0], [3.0]], "dtype": "float64"}, {"__nd__": [[0, 1], [0, 1], [1, 0]], "dtype": "int64"}, {"__nd__": [[0.5, 0.5], [0.5, 0.5], [0.5, 0.5]], "dtype": "float64"}, {"__nd__": [[[1.0]], [[2.0]]], "dtype": "float64"}, {"__nd__": [[[1.0]], [[1.0]]], "dtype": "float64"}, 2], "e": {"__tup__": [{"__nd__": [[1.5], [1.0], [3.0]], "dtype": "float64"}, {"__nd__": [[true, true], [true, false], [true, false]], "dtype": "bool"}]}}, {"a": [{"__nd__": [[-1.2761124197329383, 0.5648512404463458, 1.7960125066302708, -0.2354150525900867], [-0.3910773505445041, 0.17390146428778722, -0.3668597432540227, 0.08236522460221347], [-1.1399714593630075, 0.438257432619766, 0.39257939815601534, -2.4354174163439515], [-0.11022594532382894, 0.16291581767584848, 0.5612247602630475, -0.816089756439701], [-1.3280342642624956, -0.9507340805091575, 0.5480409641909273, -0.8518355089919037], [-0.7641438115235788, 1.1437655777004572, -0.7861575894791125, -0.8986326207536771], [-0.09123756351337108, 0.19861051640677327, 1.0376296586712488, -0.8793276811910162], [-1.5681545944352957, -0.12249665156557239, 0.6436368157294202, -1.481526543249413]], "dtype": "float64"}, {"__nd__": [[0, 2], [1, 0], [3, 0], [2, 3], [0, 2], [1, 0], [0, 2], [1, 3]], "dtype": "int64"}, {"__nd__": [[0.17892488854628408, 0.8210751114537159], [0.85922570332663, 0.14077429667337008], [0.5125052564002814, 0.4874947435997186], [0.45802147092194173, 0.5419785290780582], [0.3532328362832403, 0.6467671637167598], [0.6438345829190412, 0.3561654170809588], [0.4010066751483777, 0.5989933248516224], [0.6790625978609005, 0.32093740213909944]], "dtype": "float64"}, {"__nd__": [[[-0.24363150789234847, 0.15309789405165178, 0.11551215001106485, 0.04167949545869599, -0.1284447586887762, -0.3996732290357504], [-0.46406900048059874, 0.5781311171360392, -0.5586048574427416, 0.6140000492587736, -0.6575292179326638, 1.4517908934864234], [-0.8690554644416959, 0.8021371497270391, -0.44853607889510216, -0.05780774025334562, -0.032776285457374786, -0.3261113325383774], [1.1141963174111928, 0.04745767307913852, -0.6605368714976768, 0.32328015397564713, 0.3562008733065663, -0.5737415191612941]], [[0.2678065209875699, 0.03447789396438691, 0.4101678301340513, 0.35609773589728694, -0.004221203289446628, -0.20414143401370244], [0.4551863415733008, 0.1565996020130012, -0.7446386865664107, -0.27243458687909883, -0.24165619930422855, -0.13107725295794054], [1.1421460322718369, -0.1886436523747339, 0.23866833271303475, 0.8276666767083156, 0.2518267667430344, 0.21593878620221707], [0.6489793539651739, 0.09865534489413216, 0.09299119044331854, 0.3443692719127324, 0.20603844303984378, -0.3401787628171484]], [[-0.582945417697422, -0.056686247855501165, 0.699006016692753, -0.11320787799943081, 0.5742147459574235, 0.20629935456017567], [-0.15444851501913695, -1.3166974560844977, 0.11106682797964648, -0.13847257058193685, -0.800259628395301, 0.2572523991730921], [-0.012711258827462074, 0.18752653541082523, -0.30220794794039946, 0.10101128056860786, -0.2366159680179241, 0.532473336546587], [0.062162833238463486, -0.7012621408719282, 0.10715802360853963, 0.03382055764510263, -0.10172219953716331, -0.6748311233074653]], [[0.5297688900637498, -0.3657984599782962, 0.7913177209737472, -0.3299719910905376, 0.016238429677090566, -0.022406460221078454], [0.07558721379985832, 1.3217075050209899, -0.24914390223530136, 1.034728676973115, 0.31172911208745185, -0.3475037705244129], [0.6277802304425313, 0.420412203168611, 0.47721842599759284, -0.6669922009623339, 0.2514058025512073, -0.20125794823430307], [-0.07108279963018968, -0.06572914173899393, 0.02033417368522835, 0.7952217693384313, 0.18922009582363747, -0.9223805187266035]]], "dtype": "float64"}, {"__nd__": [[[0.6234338568660385, 0.04382155312231714, 0.06576369126792075, -0.9133300164253467], [0.1998766614275817, -0.24948273392994014, 0.4778099791634752, 0.22811410700511672], [0.4138596391106934, -0.18652590543819517, -0.9146743602628754, 0.4843963465797735], [0.21017496736345978, -0.16530989712102212, -0.27853220758407354, -0.4428056712092881], [-0.8812683015840147, -0.5886018378965101, -0.035056123054453325, 0.8047320036465682], [0.2265825704020311, 0.8023893896507223, -0.5049115889111793, 0.6283408268225688]], [[-0.20806657450867308, 0.7331646094432411, 0.6141975854635502, 0.01768140235830313], [0.06724849833309794, 0.6087173838255869, -0.3978821921939673, -1.081455443367941], [0.40846483144387186, 1.0895608440748703, -0.9480917678162857, 0.10482125348173899], [-0.5910771631601863, -0.14501003419908617, -0.5000120703623253, 0.43746055313961685], [-0.27259200808758954, -0.09642381759999451, -0.5418849810154927, 0.36644629889211094], [-0.2549368546562465, -0.18918725150521293, -0.03779913461841987, -0.7258813939830294]], [[-0.010421276308196335, -0.8059478724223631, -0.10905160351437104, -0.10987707256067654], [-0.2001820943589488, -0.09991480076573439, -0.337212291106604, 0.25538168082284984], [-0.0017083928076625044, -0.2357511384984251, -0.11461318753859664, -0.3385303717558627], [-0.06375234495765637, -0.1566841198323968, -0.9008167276357346, 0.8237719597004725], [-0.22249093171607026, -0.3357191263380784, 0.1502158755395611, -1.044624580517046], [-0.19891938788180658, 0.15316668342408826, -0.13236110250534286, 0.031900978168796626]], [[-0.22224326339259623, -0.18428738386793533, -0.02043271477877521, 0.0757770966365725], [-0.5476387411986607, 0.13942901173763056, 0.14865571406018693, -0.30943277764240906], [-0.2865155293572035, 0.05841069951184989, 0.5814138162924751, 0.5437135615824399], [-0.41609988166305545, -0.2710212487687104, -0.28128342621657865, 0.19021614344543414], [-0.08078786225312748, 0.15855606906957787, 0.6215267044226278, 0.10473581634736936], [0.19624493471088805, 0.16596442431525202, -0.39147516365816654, -0.08849634141467332]]], "dtype": "float64"}, null], "e": {"__tup__": [{"__nd__": [[-0.08497972733349436, -0.2619449420284318, -0.292808623801616, 0.28595213448913237], [0.06201663719604052, 0.10139079380672048, -0.06682453058259763, -0.10536076827619359], [0.3353573758092245, 1.0617186804373917, -1.2960442893403807, 0.7014700941606644], [-0.2868733099439912, 0.10257635974914578, -0.16644525183306152, 0.008581823872569241], [-0.37190333711258583, -0.7210191161553335, -0.9427071030490743, 0.6988911423124221], [0.2709624671518779, 0.8087959227910458, -0.6637761818396248, 0.36915331927963035], [-0.09307009414616407, 0.13822950136620582, -0.16532634928977075, 0.34120886084369295], [-0.2237436976413412, -0.020875420587684587, -0.15311169337475766, -0.5974127528761972]], "dtype": "float64"}, {"__nd__": [[true, true], [true, true], [true, true], [true, true], [true, true], [true, true], [true, true], [true, true]], "dtype": "bool"}]}}, {"a": [{"__nd__": [[-1.6352904120607807, 0.09352351648286154, -0.038902375054806385, -0.4810858401814653], [-1.367976003986649, -0.3494670858987591, -0.16312948980393505, -0.010201615660946052], [0.051451314536709175, -1.1997401542189938, 1.0219434439348336, 0.7933131780938874], [0.7897362009528284, 0.46730773920614466, 1.3174075616613432, 0.7322071794100257], [1.1285684978984905, 0.3272509980638429, 0.1460032681679747, 0.6342519952864688], [-0.7023747010885814, -0.22598188251227408, 0.20759682628912152, -0.5880510333462579], [-2.4449376260759905, -0.5836718826704723, -0.7428556249881406, 0.8602678354688927], [-1.9085104330181148, -0.7701187787970525, 1.5907296143546934, -2.3668022282306573], [-0.2768291529543013, -0.6473647091070105, -1.7230433291680791, 0.0009138179660268217], [-0.8935764143864541, -2.7128996561786796, -0.6177052445789465, -1.3580076584977345], [1.070989234844979, 0.004066286868374917, -1.1174064522241616, 0.1938997174850802], [1.895946176525527, -1.3006847488714992, -1.0197238125799062, -0.11523996269018963]], "dtype": "float64"}, {"__nd__": [[0, 1], [1, 0], [2, 0], [2, 0], [2, 0], [2, 0], [1, 2], [1, 2], [2, 0], [1, 2], [1, 0], [0, 2]], "dtype": "int64"}, {"__nd__": [[0.735747617649181, 0.26425238235081905], [0.2097535781809798, 0.7902464218190203], [0.3949228351499696, 0.6050771648500304], [0.6181140321258596, 0.38188596787414053], [0.715205442610459, 0.28479455738954096], [0.5514644082494795, 0.4485355917505204], [0.5028622061301566, 0.4971377938698434], [0.7095370691540273, 0.2904629308459727], [0.8578560249716278, 0.14214397502837206], [0.38113017499739926, 0.6188698250026007], [0.2879744525015309, 0.7120255474984691], [0.42672272088096735, 0.5732772791190327]], "dtype": "float64"}, {"__nd__": [[[0.01724765204502102, 0.07657722337173826, -0.7606789470557579, 0.04514109978005477, -0.5939802835331094, -0.3985966304816024, 0.4459252311995229, -0.37317459382222173], [0.28949584762075253, -0.17618865124251862, 0.0711837619978699, 0.06565040854752856, 0.3085930214862539, 0.3554146435140171, -0.3795119687078528, 0.8194616728607411], [0.2932444128653646, 0.4691588691479309, 0.4898888427169032, -0.24186465036495894, -0.0349637506133572, 0.03856077985371346, 0.14013202486719636, 0.8465724486232382], [0.9262566166478257, 1.727319188073138, 0.9182194946234584, 0.8439367972500337, -0.032168819098026415, -0.41558909207561656, 0.012823770431126995, -0.13720353734578314]], [[-0.5001701979350355, -0.11781705796477979, 0.22009895150390854, 0.028702591601462935, 0.455401968021771, 0.31355024196547615, 0.22125589148026467, 0.6377002023117907], [-0.04722657023706725, 0.04537521361569455, 0.7374193653485134, 0.48737583503300597, -0.7441371774159539, 0.07010813492944903, 0.7797021800918733, 0.24940101712191481], [0.6246142064460679, -0.7435202371711552, 0.23633832394240356, -0.9924203268973109, 0.3703055025723674, -0.5940741394914222, 0.7236258731870353, -0.32341471223486007], [0.08022041772594665, 0.15705938713754491, 0.010784351017298388, -0.7507376164260606, 0.3814929760790935, -0.23132781319431717, 1.2433917574930509, -0.11334775912860404]], [[0.1294258124394416, -0.14530043020615854, 0.8817990525596615, 0.770107413197401, 0.3272380115803709, 0.5407796410420601, -0.42057681598923813, -0.45474136074396054], [0.6688480739850062, -0.4113041290080156, -0.13281765113454067, -0.7582695906190119, -0.17722638227621676, 0.22374741899394557, -0.023740132430026442, -0.7466402668284632], [0.41263955140071407, 0.09909634335319477, -1.2631208415195927, -0.5372282609758631, -0.16462760069758406, -0.2595066860249234, 0.08740681246803442, 0.545592772277072], [-0.03431579712934655, -0.04028194424056914, 0.6845821797143449, 0.8077626882994993, -0.1506537026132922, 0.8155234983644483, -0.8725187073158958, 0.7883787854987703]]], "dtype": "float64"}, {"__nd__": [[[0.24572630580910088, -0.4183167482749177, -0.24594218079256497, 0.07009318040493928], [0.671073844749054, 0.44051871482325355, 0.15305161131570996, -0.18810207609179896], [0.05015751409833107, -0.602282591888437, -0.20037411094901642, -0.4270100259350633], [-0.1806941325705085, 0.22010360120714204, -0.16460896358606716, -0.8908118064760029], [0.5367764108193647, -0.014146755003029322, 0.20594168044289632, -0.3629853091785972], [-0.13724453845143053, -0.1570861893609967, 0.7002740215145289, -0.6278953669834615], [0.4285175405456836, -0.05593337238716691, 0.33628786980453007, -0.32509459201778657], [-0.846824235014298, 0.4399368969743821, -0.2067586512073701, 0.2188302802897135]], [[0.332308797190976, -0.8863825373672513, -0.2707139890618187, -0.12052001966040242], [0.267381314697968, 0.2835855971238353, 0.3736994992827134, -0.1251536342950073], [0.6626862063918098, 0.007108174196407166, -0.042015402800506176, -0.2534104618208141], [-0.5735279580404424, -0.017304631396871512, 0.12413997666206456, 0.13206854194417844], [1.0817116282478934, 0.14610467247572712, 0.18082419676100858, -0.6557869340616235], [-0.1314419873508385, 0.008772634241853594, 0.23116953477427468, -0.10712825920730769], [0.16515235148737073, -0.5020971680021448, 0.5572997213470269, 0.5062144747050198], [-0.5772147115722257, -0.7627141107508564, -0.34590417065009454, 0.06743013631340247]], [[-0.4539912014046394, 0.06783134599243822, 0.3612804081367953, 0.47227282082798056], [0.5399276231481153, -0.7469949963213682, -1.0102541299551822, 0.2308110712459664], [-0.4035550756077745, -0.40000287611713015, -0.5605449304420932, -0.3440243955468736], [0.24522906290743604, -0.7539772268077176, -0.7101819616128106, 0.4713100386243245], [0.2789328012777315, -0.8835804001228967, -0.9631198588076556, 0.13369207106879738], [0.36981947220893874, 0.9703274250657382, 0.2726482857847625, 0.26277575674742787], [-0.18685196228785247, -0.034367864236012534, -0.2738690656254398, 0.5185821139362347], [0.12241003917156026, 0.46357062155917145, 0.08807069218213044, -0.5347978225539156]]], "dtype": "float64"}, 5], "e": {"__tup__": [{"__nd__": [[-0.1069814557465897, -0.22963476626564455, 0.38334328283321045, -0.8119757870935773], [0.290806983288676, -0.5659265382565152, 0.17338293351278034, -0.7313605778353859], [1.4395242050139534, -0.045001547265567515, -0.37785832935640057, -0.976978244018829], [0.2668880472454038, 0.7703043398212612, 0.18733324586138, -0.055586630328734815], [0.05306922151463428, -0.20712128917937153, -0.8185336212433318, 0.4331045573217294], [-0.005189335640204568, -0.0791076105455546, -0.2522618676396487, 0.22775378123252074], [0.2705529177419158, -0.24597045718509886, 0.061882111371206285, -0.11157141036196538], [0.4231585490020903, -1.128710258481543, -0.3447243641285032, -0.15346893334241282], [-0.2605039614512329, -1.664119403071872, -2.0403432073185797, -0.012272259588908421], [0.3172621012706981, 0.04453146226648462, 0.11586419494450984, -0.2250503895991903], [-0.26902753025982723, -0.16118842882554973, 0.08391908240796182, -0.025571019604424584], [0.20692786719027728, -0.014466675963150419, 0.160935252379616, -0.22259302129585792]], "dtype": "float64"}, {"__nd__": [[true, false], [true, true], [true, true], [true, true], [true, false], [true, false], [true, false], [true, false], [true, false], [true, false], [true, false], [true, false]], "dtype": "bool"}]}}, {"a": [{"__nd__": [[1.7848415861196845, 0.0584113607284587, 1.0632838769435624, -0.39558586293782894, 0.6994031154090845, -0.1607797079332106], [-0.3552517992377278, -0.16999131286838967, -0.9042177129459059, 0.41160483767154743, 0.3295886517045294, 0.6249212863420751], [1.2027095236453667, 0.9799156123089097, 0.4041544709279531, 1.007681265227766, 0.15713453390812346, 0.9318086513389016], [-0.09765045319617155, -1.7889283734787247, -0.716145924420419, -0.9550093464082494, -0.46897908199651517, -0.14893906178732433], [0.10439618164490383, 0.5301694996940745, -1.034683026416445, -0.44248455192111247, 0.6812386259501284, 0.6124381445307081], [1.2468270566492123, -0.23438907863941383, 2.231030408587067, -0.6471105962038151, -1.495080540370972, -0.3429053840986888], [-1.33365083110837, 0.6275152595252322, -0.119103695907185, -1.5156625143330753, 0.1492796548980362, 0.9590493422725859], [-0.6160429994281211, -0.42514326630419985, -0.7881735721633307, -0.6216293098806679, -1.3944572112048434, -0.2882970028259174], [-1.096614556201501, 0.0933564809861979, 1.0566163797544474, 0.68916879886491, 0.2110764986268949, 1.0370559824577665], [0.5670418041683455, -0.6472184256730338, 0.33608087136860704, 0.5667043423598254, -0.30101255523742476, 0.526388260302557]], "dtype": "float64"}, {"__nd__": [[2], [1], [0], [1], [2], [1], [2], [3], [0], [2]], "dtype": "int64"}, {"__nd__": [[1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0]], "dtype": "float64"}, {"__nd__": [[[-0.44694977509314165, 0.16254383347565268, 0.26415415855500396, 0.05756470111921007, -0.16605301941700606], [0.11018268174321827, -0.44238822569197933, -0.30281634215098396, 0.24952781164162047, 0.4720603453600154], [0.3952394937798373, -0.03953196472125528, 0.209970063011208, -0.31723602080319757, 0.10009091265877365], [-0.318567869908413, -0.6016318968482607, 0.14713294844264738, -0.17259250141682314, -0.10694295756316549], [-0.41836984523108606, 0.7444023161148904, 0.21909883617731069, -0.4061190110670684, 0.30937010509542273], [0.017834722749417865, -0.3740701149181678, -0.3902297462433545, 0.4630379889453133, -0.1613167512465542]], [[0.19217162958624345, -0.31209132357880925, 0.23703038629902665, 0.21997520081999364, 0.2343475552456393], [0.19786701463837134, 0.497425074675817, 0.37736271999544124, -0.28104749389780054, 0.8247495352773367], [-0.22522137601200237, -0.29176873427131084, 0.009518610614729863, 0.4132479368780423, 0.23879920024306567], [-0.530185321307528, -0.22599142641857625, 0.4747808204590712, 0.12914161827383722, 0.12103770649541776], [-0.7655845481357307, 0.014691842651205484, 0.13565704163918083, -0.5214876500693505, -0.9121156904506814], [0.10949691892521421, 0.3942856734286309, -0.6497106656598799, -0.1607533150044293, 0.6662265760282762]], [[0.7884466707531964, 0.054918773784689456, -1.1124313849741876, 0.08585725926730918, 0.4581700671359331], [1.2592517314559697, 0.5120393185195417, 0.38584143532182097, -0.12086103212622308, -0.7044938401782496], [-0.37947201924964263, -0.4171797745314635, -0.2791174497514853, 0.37782244452535924, 0.06194823349408528], [-0.9858325466713673, -0.15531192401897895, 0.5978714984892703, 0.4168241905499444, 0.7565669563948797], [-0.5252783108209996, 0.19854818691236512, -0.16081169396512265, -0.5242562336798761, -0.34014877112229025], [-0.8755458987280713, -0.815693050047209, 0.08291967239857001, -0.09154062023304296, -0.22039803306892525]], [[0.6250875702581312, -0.17970482168267146, 0.2848038523899315, -0.5353404074242404, -0.588133820426138], [-0.22397601368566147, -0.2766038266867776, -0.15377343489258366, 0.017948822912580254, 0.8324392325228561], [-0.45347357385503456, -0.6287671718031236, 0.24724245480201812, 0.6114790946622636, 0.8689722317006656], [-0.5298053515125957, 0.2549974066859384, -0.8917740814687777, -0.03127073413889755, -0.5343937478823457], [-0.2723109387322201, -0.7312340939428502, 0.6233352190866086, 0.4706741113850146, 0.23469675726887237], [-0.46780737778995324, 0.8347071348868718, 0.5293657958817614, -0.12127541027791347, 0.5459709235483761]]], "dtype": "float64"}, {"__nd__": [[[0.2531434716845946, 0.31422654251782456, 0.4026970528722146, 0.39058642519579995, 0.010988266302324202, -0.6421473230398475], [-0.5071385752786943, 0.22703055467254665, -0.20375405162183832, 0.17311990424103263, -0.9095937681163557, -0.2048857806730397], [-0.19840459734133684, 0.7608376434474208, -0.13865414385166036, -0.3028393569734842, 0.05965558268253464, 0.5997350675671751], [1.2747997780478886, -0.6674442147089374, 0.875313736180752, -0.9619426623056299, 0.6802594483675909, 0.7293936424651836], [0.5648370749756313, -1.1147907822256733, -0.46829492433476244, 0.8978782903229412, 0.23383662054631565, 0.5325846269479546]], [[0.0343128353101438, 0.1589582484625137, 0.17640791739118317, -0.3019886496901298, 0.49327432516222935, 1.1622027748705346], [-0.5313241550364509, -0.47709310023769913, -0.4749168340519236, -0.40329940275953735, 0.08123803617620351, 0.24086745964918446], [0.2760040389809815, 0.12695214791463158, -0.6013764556417353, -0.07475497299053904, -0.5121215767467568, -0.0620378397018903], [0.2500100345776003, 0.03378065588418848, 0.656249703193852, 0.06827011087527288, 0.5312726734836084, 1.4435648111115578], [0.414164148816266, -0.7656885388789421, -0.5939215374679407, -0.06664565265605134, 0.39428206967038065, -0.1800961774522634]], [[0.3118388518545902, -0.22057020010753542, -0.12563232436765318, -0.12352084798584012, 0.9877934702080722, 0.4766221752912631], [-0.6751232926922116, -0.05314496415522998, 0.8291525030138123, 0.06979151432567392, 1.1605141505582597, 0.14511733677739572], [0.13239454743769236, -0.13395068399442464, -0.07655096875488529, -0.07156463325873758, 0.31247951193529183, 0.6081692828058518], [-0.1462019518679675, 0.3057135736291648, -0.6226513342519223, 0.0635758599679445, 0.31324886102027805, -0.024640912324052668], [-0.37465974673839897, 0.3243591915934892, 0.5252311307377194, 0.6250688659995398, 0.29897788224430166, 0.9109427485841031]], [[-0.12032121125450444, 0.2990090184352987, -0.7641372138602558, 0.010911702011991974, 0.05400499913346464, 1.0834960934783198], [0.6595495917979167, 0.6236941733467217, -0.46441965931580037, -0.12580560690506368, 0.09374556514185249, -0.6164467136799835], [-0.1929905278215643, 0.3835653537087915, -0.015533372582396435, -0.4353057527985646, 0.9643746009983689, 0.24439397908890534], [0.006115145095314864, -0.625642216029227, 0.45757184895525654, -0.12087879694354009, -0.21734137222921954, -0.3178863757358359], [0.6345779697060161, -0.9228718038827619, -0.5083827006986645, 0.4033600123114941, 0.17895966948763478, 0.2534593367662319]]], "dtype": "float64"}, 2], "e": {"__tup__": [{"__nd__": [[0.24402809451058105, -0.15448102085595827, 0.016841070808040426, 0.06280472579622587, 1.355487647717972, 0.9032501279661056], [-0.23822244234822812, -0.21390761644991574, -0.21293187416328893, -0.1808217597718056, 0.03642357157306586, 0.10799440224189603], [0.5364990261243665, -0.35776238783787223, 0.28802792220222523, -0.28056702862610794, 0.27994530842280135, 0.32661774673645017], [0.10451323480148003, 0.11252071367011425, 0.3293886764668374, -0.16998977509343796, 0.49012178351931374, 1.2181810984552748], [-0.058348986739589576, -0.1802129822259326, 0.2525142710369191, -0.05959230965100601, 1.1728121896898898, 0.4191107160215872], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.7770245929676518, 1.1110372060210534, -1.3208700724468039, -0.1591867474858633, 0.17525643489276693, 0.15884986640785945], [0.24738846967098607, 0.02338453577036525, 0.1800046493600151, 0.38582585935941305, 0.04343993506967673, -0.320497167254725], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]], "dtype": "float64"}, {"__nd__": [[true], [true], [true], [true], [true], [false], [false], [true], [true], [false]], "dtype": "bool"}]}}, {"a": [{"__nd__": [[-1.4183657336563467, -1.246525422535126, 0.25895135145280396], [1.649215289337211, 1.5698483045602045, -0.44108660479055445], [-0.893711704993485, 0.7718193862678931, 1.3606409174675451], [-0.7563948258708804, -0.6894089907475883, 1.5125553236491567], [-0.8224709651347768, 0.8500032001034983, -0.7107597434114393], [0.43593190653461067, -1.3219642465179617, 0.48956053880142697]], "dtype": "float64"}, {"__nd__": [[1, 0], [0, 1], [1, 0], [1, 0], [0, 1], [0, 1]], "dtype": "int64"}, {"__nd__": [[0.3649468302198841, 0.635053169780116], [0.3022885519535192, 0.6977114480464808], [0.8357806643791952, 0.1642193356208049], [0.4318909587730218, 0.5681090412269781], [0.5264050790939739, 0.473594920906026], [0.5377397924464706, 0.4622602075535294]], "dtype": "float64"}, {"__nd__": [[[-0.5168639985992287, -0.596774071143696, -0.25640972997547234, 0.6302632335235485], [0.17305584768920554, 0.03015381090232679, 0.3966787438208462, -0.3022390394527665], [0.5704357529182665, -0.6530905408834363, 1.022498157412087, 0.2509210570744459]], [[0.1124466607166656, -1.1552932843080805, 0.024212408680611065, -0.0014668416843146514], [0.36956244239024544, -0.5423418151637962, 0.8603535706529926, -0.07482019977617661], [0.18074668275333122, -0.27412894636873675, 1.2607229400718671, -0.26974651457775733]]], "dtype": "float64"}, {"__nd__": [[[0.08112241849591928, 0.5130619992194231, 1.205333014744681], [0.04820539579448804, -0.2241542001144846, -0.11138810272460589], [0.07497900672782636, 0.34970643352100006, 0.06447205754781878], [-0.5059190209748978, 0.06663632155741996, 0.12247971293356687]], [[-0.23626550914170036, 0.5057131874794996, 0.6803099909788775], [-1.145647449387858, 0.3402385029874554, 0.3737775055944338], [0.8140691925815661, 0.44968930830546444, 0.8644341693925623], [0.6168707540820524, -0.431164264573833, -0.22431779590740844]]], "dtype": "float64"}, 0], "e": {"__tup__": [{"__nd__": [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], "dtype": "float64"}, {"__nd__": [[false, false], [false, false], [false, false], [false, false], [false, false], [false, false]], "dtype": "bool"}]}}, {"a": [{"__nd__": [[1.2838635965607184, -1.3609403502895836, -1.2233373002875008, 0.546070848649934, -0.5326452133028694, 1.4027737709852899, 0.8289720009943885, -1.0514241333154095], [-1.3545647506926906, 0.9297327247231177, 0.3715551586812507, 0.3992440441052236, -1.1592120042954672, 0.3117071078489212, -1.4579070809047256, 0.37525488589904], [0.531834395921179, -0.7634512909216852, -0.6726502785213637, 3.3184837576204593, 0.25733061265227425, 0.08743044803443618, -1.0930159686265832, 0.6263253376114251], [0.19150174441970216, 0.30902074781043837, 1.2423269238104844, 0.5986336236395807, 2.6338738678381204, -0.5056797977808716, -0.7244844838044957, 0.9446880820983399], [-0.8365076589795607, 0.8987284069162764, -1.1886855912845533, 0.11070813283062962, 2.1581816133598486, 0.12118431489116613, 0.46706518638484396, 2.006358106329647], [0.6096056259436389, -0.4199410737075394, -0.11656315077956339, 1.7505368108725297, 1.6551339617858762, 1.5293730721829628, 0.1462725822015197, -0.6231218318933468], [0.3926235820389074, -1.5560981097737268, -1.4180637662890003, -0.7127766113739142, -0.047565296113321824, 0.5771815942896593, -2.3029104123003292, 1.8389060439170157], [-0.09475699982178676, 0.839663752000987, 1.7494146228332095, 0.5736213957154324, 1.9473991313662204, -0.3282386760075715, -1.062883715422249, 0.06305088406163342], [-0.8376298491383021, 1.6071357102949493, -1.9693002161327284, 0.5493793766560487, 0.33725392194366755, 0.45711163031702196, -0.3713542867118799, -0.8161616838937953], [-0.14575488691041, -1.7392166693008593, -1.1813910584776588, -0.49981306769922473, -1.0596606485173299, -0.6839905663906842, 0.4763261790441618, -2.4976403960899702], [-0.8943396196632339, -1.6902570297641477, 0.2987894818618129, 0.8925885320771365, 0.0973772465765271, -1.6301598808539077, -0.6702303740606248, -0.7740590956045297], [0.06950987289111885, 1.128321346026966, -0.5373690600598151, 1.7600610530662395, 0.4974062510866995, 1.4093478855176784, -0.33416611267526874, -1.4540830750026716], [2.8784616625087747, 0.1280789678846594, -0.7253989238373454, -1.4569983373672866, -2.240944441059077, 1.509854163043201, 0.26879808379434844, 0.11134627370853796], [-0.7228424864784906, 0.46649662210757215, 1.1863116623472707, -0.9204371211909348, -0.4259212751069976, -1.6790116431780988, -1.4303760083781623, -0.5999468132206068], [0.19349362157399483, 2.2207040300918335, 0.5665575624228096, 0.3759797810692001, 0.10512579604953468, 1.0587631229278798, -0.23062518380184824, -1.0189875618027922], [-0.6944846477065086, 0.06533337089919351, 0.6574197382323792, 0.6545246889490558, 0.3167546543307997, -0.08966409958236884, 0.7863269505043972, 0.42199024506073235], [1.6064700314261604, -0.4835177878191216, 0.3579826443615271, -0.19547105120763586, -0.45606776729719267, 1.1411367467126534, 0.5078539408990458, -0.2897315183231543], [-0.14297404875628814, -0.526590575066458, 0.894310034595299, 0.3436379553372577, 0.6359138001471567, 0.18180368720026593, 1.125241703285776, 1.5489987000573293], [-0.5745829712641748, -0.517547855996593, -0.01505892883901581, -0.652653145233475, -0.11500863013183513, 1.0635382249935759, -0.5594760748106198, -0.9249099964615192], [0.1277876298512573, 0.9130098695777894, -0.020942835066497117, -0.9790565191124933, 0.4502406779670835, -0.08436801274031265, -0.4051867190004839, 0.15551487965457841]], "dtype": "float64"}, {"__nd__": [[1, 2], [7, 1], [6, 0], [4, 6], [4, 3], [0, 7], [7, 2], [3, 2], [2, 1], [6, 7], [6, 0], [7, 1], [3, 7], [7, 3], [2, 3], [3, 7], [4, 3], [4, 2], [2, 7], [4, 3]], "dtype": "int64"}, {"__nd__": [[0.4441973674116097, 0.5558026325883902], [0.6842144525165601, 0.31578554748343984], [0.3241816369037674, 0.6758183630962326], [0.4166336334755297, 0.5833663665244704], [0.5574429081818693, 0.4425570918181307], [0.7734268068625377, 0.22657319313746224], [0.47195434471629505, 0.528045655283705], [0.6806979406294414, 0.3193020593705585], [0.7513078317634714, 0.2486921682365286], [0.3347285415054914, 0.6652714584945086], [0.7293176096359192, 0.2706823903640808], [0.3130757987006376, 0.6869242012993624], [0.5300331286504059, 0.4699668713495942], [0.6946295158228776, 0.30537048417712237], [0.5315292138905309, 0.46847078610946913], [0.4859536821637463, 0.5140463178362538], [0.5349099809976691, 0.465090019002331], [0.306926331097284, 0.693073668902716], [0.46174298970762273, 0.5382570102923773], [0.6009102235925718, 0.3990897764074281]], "dtype": "float64"}, {"__nd__": [[[0.3045404293172581, -0.26644609636762356, 0.08416391545708532, 0.2605335686899964, -0.4323638792196206, 0.012034293687439222, -0.18143435650305006, -0.07203949299088991, 0.40983371808814645, 0.5191996403626108, 0.14749292678058737, 0.6703008142539664, -0.4679307392096791, 0.3989427620970085, -0.2507427499648774, -0.2278457717408025], [0.7841895438210542, 0.4949484226223638, 0.7229011472049479, 0.8132679827424761, -1.0961009997725963, 0.49348642145367116, -0.4861895919983248, 0.4953692902337437, 1.290794840601545, 0.023154035828235428, -0.9731141599589138, 0.22322229874496202, 0.008892230335546928, 0.825507678738063, 1.162623794560526, -0.0006890896278258609], [-0.5888531695851574, -0.4790213951020273, -0.6786205285397955, -0.704726457712789, -0.21228115434670075, 0.22682206159897753, -0.5479531676453692, -0.07771509459384188, 1.4930943549732816, -0.871879668591874, 0.34586630158650566, 0.11691069898519059, -0.09102308249187627, 0.3090847059479179, -0.17863195163983814, -0.4674397853294921], [0.13299642691358776, -0.03844315162113102, 0.29594746135557176, -0.5353864212503753, -0.5625388701730657, 0.8434948125420945, -0.00996987527533971, 0.3204141979130521, -0.2841982146975873, 0.046100897434628585, 0.025261989636656573, -0.4260300960031117, 0.3310133050997752, 0.6494478378264381, 0.18421795653036763, 0.5637147245599105], [0.3950212670989994, -0.054187581974176294, 0.08056034597358551, -0.5219395295743349, 0.12528260696100635, 0.0374917136335729, 0.6915041230848806, -0.5482315060793524, -0.617591201252523, 0.30649526559765, -0.8923274805965997, 0.06447524510856306, 0.4201457414746536, -0.7619187115152743, 0.2516565754780309, 0.12345735450534713], [0.5804349602755303, -0.440527653967639, -0.28002760151201406, -0.3873758276371987, -0.4613227460555601, 0.4468644615127111, 0.4166889273165612, 0.9312637809549125, 0.6726366046960092, -0.657441149827501, 0.03652503907198872, 0.33431711568751166, 0.6310274873288939, -0.1448856956601214, -0.7913361171210583, -0.2863321162732145], [-0.4747132486320301, 0.09402183788007079, 0.48779094382357374, 0.6517913928683996, 0.0963992791055991, 0.1702805433849784, 0.2799789896887382, -0.16821922157214172, 0.37847546584704683, 0.9668461089083998, -0.4753564824711522, 0.6323618983743128, 0.47937037381032516, 0.5591163686391232, 0.6416639137495582, 1.8221283858091812], [-0.02120822236029136, 0.11643123769357097, -0.5607396813828615, 0.019611136765349188, 0.448306910355823, -0.22967074370249332, -0.40311525666187503, -0.33522794336772843, -0.7344907582654653, 1.2634201212650322, 0.23272327738589751, -0.563221143989566, -0.024533018170123613, 0.22135448381148634, -0.10914246089523134, -0.15258233064210433]], [[0.21664944547611856, -0.47536615107243346, 0.055551745595084114, -0.4170697959201518, -0.506162968979471, 0.25545152295433504, -0.445751023688671, -0.2587366064035927, 0.030483881263499633, 0.5498951750623826, -0.10874668525299075, 0.34272799496054973, -0.7078411600662211, -0.5098789879808326, 0.8575548606337311, 0.08482328771346624], [-0.14252019358616222, -0.5381918304042955, -0.1907660236124513, 0.17172508820300575, -0.0851372340714797, 0.24457882161107394, -0.12153637623345499, -0.41224868350515254, 1.3311673222562268, -0.06020586180684252, -0.00911923875432397, 0.20907248874347567, -0.006956422196104502, 1.2293017254771244, 0.19537290537561045, 0.46344223138220636], [0.14535789450344053, 0.8447210600346318, 0.8391482962295154, 0.04542491337414944, -0.565680830197419, 0.29525211336271867, -0.9516359314065139, 0.0757973757813664, 0.5614263322390951, 0.12177348178263675, -0.41845175310841193, 1.1131653233662626, 0.4204053129369104, -0.45411860386905056, -0.8182195887628563, -0.5387816834451182], [0.5479099554494884, -0.09708029168973857, 0.3437130975249973, 0.04431487833061582, 0.29114070104736195, -0.306931054978165, 0.8063057093267654, -0.14884358906684012, -0.04348643080859991, 0.19662517644173677, 0.39926611249951743, -0.0597191450082657, -0.22476713409573904, -0.037600642829635926, 1.198562331842217, -0.9699736015004171], [0.300288932650845, 0.12625147821521385, -0.5315874006618011, -0.5889814850286943, -0.32924727816099564, -0.6254991025075869, -0.1335165032222035, 0.14638992127931844, 0.10335864959753678, -0.8612027920679902, -0.6795944336915691, 0.9037721701654376, -0.609143471178451, -0.3570433885706739, 0.742808415306201, -0.35887263017375504], [-0.03006925733168653, 0.5501769040058716, 0.7070540887023765, 0.31663864025260785, -0.2634218968079146, -1.0969496028993941, -0.2760042517955882, 0.1594058691592755, -0.17225078923093892, -0.16221067323128205, 0.2741173341446589, -0.29706379364478097, -0.731617603408502, -0.17733902161464385, 0.054601036405759217, 0.3133798825363407], [-0.08920873176740697, 0.015810923466975924, -0.5112668983995811, -0.2408885650421461, -0.29646885473589346, -0.42457073086122776, -0.16576417732597634, 0.7243479747549157, -0.22726791622133585, -0.006375572528639632, -0.9411262615994616, 0.15513310629648921, 0.6520382246893863, 0.0497068812709731, 0.21901579575897248, 0.2595800045932133], [0.5966594804440069, -0.052250099003855896, -0.04250469797225082, 0.4565949847495979, -0.23762954500828185, -0.1950057596960634, -0.18417281857397977, 0.10255710958709219, -0.14582871726410804, 0.6942776440675517, -0.06809184229215813, -0.3518780700339557, 0.6872034598521924, 0.5531203982990055, 0.9332689495585569, 0.42301084864003485]], [[-0.5226500945462479, 0.8497651005469259, 0.49180897660538364, -0.4818303952641123, -0.18336041071106432, -0.7072012252715427, 0.10112902481953848, -0.5424967801741122, 0.2376650739094264, -1.1637754407172705, 0.5357114073243088, 0.7690686350965454, 0.1216247362155026, -0.6687061493305723, -0.48278784071066194, 0.33570094762937336], [-1.3631482742166892, 0.31449563443023076, -0.7922839499509541, -0.2879782532631378, -0.3326684651935439, -0.057734222007861526, 0.20604315058550368, -0.3068365243241322, 0.04547458193599663, -0.7794637229724993, -0.03723530785805383, -0.6866022453196886, -0.07098157141573418, -0.10955900774091702, -0.002192438247781304, -0.014970405918889755], [-0.1436265332915766, 0.32083650717472817, -0.4025334515556336, -0.05795469155449445, -0.10778015795401162, 0.34758346427194753, -0.12780681287035275, -0.29557407322429047, 0.7494579432884524, -0.2678879218770667, -0.7522028109986053, 0.8055234297196318, -0.2163666563278456, -0.02931938192299576, -0.0930320533476222, -0.08660373691575382], [-0.2090521985482618, -0.8957464707997651, 0.04705906923089315, 1.0891879614447846, 0.3983643566342686, 0.12418134434438347, -0.22965717591946133, -0.5937897023944518, -0.4508544810752973, -0.4440229887538567, -0.6381739428143851, 0.7046267490754446, -0.010353797646856773, -0.3810332532854709, -0.07722727375456256, 0.7757935179795105], [0.339099271000487, 0.5654637856389575, -0.3686335899814872, -0.6907254316556085, 0.07685412736510663, 0.17175736748794476, 0.44086734817810974, -0.6671865413338274, -0.14991286252365443, 0.8312458704291915, -0.7302089029028984, -0.19634091435559264, -0.1429287632206962, -0.558944213795921, 0.4826663849989916, -0.09014344949263373], [-0.6769643725701087, -0.4278292474453137, -0.3806047042839372, 0.8714031021952136, -0.17706128148200226, -0.3338996463511951, 0.6970025319066487, 0.15838215541850784, 1.1655196059411157, -0.24836623577451836, -0.9077806568160397, 0.7286687002954493, -0.23658366678313061, 0.012559154158304416, -0.06835263800853333, -0.3130443017536426], [0.3845845495771805, -0.3653107248008081, 0.23006347740613775, -0.1297460151561682, 0.32070680073753555, 0.9506628966451341, 0.10120458152623786, -0.15431091418112855, 1.383235386345094, 0.958463447014901, -0.05459547287383361, 0.48551054237009295, 0.23929888973006763, -0.08225968512052427, -0.4096831121347503, 0.0896130748209814], [-0.559520435741742, 0.799412693488907, 0.8667495657372999, -0.37553885715023744, -0.30455079055718387, -0.5972886614725128, 0.6750826550721023, 0.06416081425144855, -0.532133535378425, -0.15206755467995456, 0.5219864614864693, 0.18325066080627359, 0.5976624992725543, -1.0852799307682837, 0.1920081314889023, 0.2519340824517737]], [[0.20502292000779704, 0.16684165470797196, -0.24132281211635262, -0.08110636618527126, -0.3725011773778781, -0.6592706542542456, 0.8118906321193504, -0.3860259803177424, -0.2750728255381199, -1.524652723712317, 0.3424022241611791, -0.11296008549114799, -0.27548914713117634, -0.12266742353613591, -0.14981750914712866, -0.375762755320229], [-0.35553375638817253, 0.2332786092145164, 0.48703885381738826, 0.5564777124894419, 0.040798294752148155, 0.6296196386471825, 0.5738566137034103, 0.6750658042360917, 0.5106453319685447, -0.707914773092779, 0.06096737157613953, 0.23121883609054214, 0.800756791304573, -0.20389861232885428, 0.26883878262979355, 0.12071467585028609], [-0.7528280687252358, -0.24103143941522373, -0.5163394298184513, 1.0659208502374735, 0.2537922653234607, 0.6351773594417001, 0.250514802202018, 0.3238719832340451, 0.6323596106470356, -0.22178901737276188, -0.49081301849194525, -0.39241458453702577, 0.5642356474778779, 0.05947462090994537, -0.38183082453744494, -0.6208263810530725], [-0.2706583601170973, 0.21372773252563357, 0.02175803510900404, -0.2748840710063332, -0.4921970297157348, -0.21451667147463124, 0.7777208399137379, 1.2363408741491266, 0.14548842981510388, 0.8226693647687379, 0.09373722276531235, -0.6124210583981768, -0.1383216160282004, -0.5989278814406502, 0.19530182556098014, 0.06089815244316659], [-1.0313383895416939, -0.49189079177115974, 0.8989899942507893, -0.27808114449928306, 0.08081057021971609, -0.18576557815518213, -0.41583751186556506, 0.13856937987378704, -0.25809509601999286, -0.3403295023429543, -0.06959103460778358, -1.0031965651720396, -0.9048211611467748, -0.9721666428404852, 0.551769193172212, 0.2244912957531277], [-0.015912851854212588, 0.20898449782052173, 0.35533514116599013, -1.2586284849155487, -0.960602300393016, 0.05285066768900765, 0.4831787714676099, 0.3177044034077304, -0.7466659917025096, -0.04462917536407055, 0.4194257541688354, -0.054291355037313825, -0.3736974045413001, -0.3229646669491734, 0.5161788168714122, -0.7205285083133679], [-0.34567271745953276, 0.45628535013832294, -0.41123179684938044, -1.4491548338780116, 0.16584251327327126, -0.20451581522063514, -0.806252152819246, -0.7127654574747055, 0.2524439253970705, 0.39513671465526806, -0.5992023829861332, 0.21636486187387535, -0.18726919866223918, -0.0071755915715620115, 0.387896399264448, -0.8056581660332024], [-0.3145709076047256, 1.5399073660461255, -0.6620025906365113, 0.08453836034351166, -0.5805461594940636, 0.41919098172852653, 0.21410905157716015, -0.3921998886405941, -0.026562571093954292, 0.2579316200443442, 0.6223785627596033, 0.2334365543624865, 0.6732574052668492, 0.1349930661051239, 0.5822797300442115, 0.22407836518256935]], [[0.09666246452564266, 0.28405277178145516, 0.2864386671487955, 0.06624576283135594, -0.5012856670537597, -0.12561389608188847, 0.6069679884720218, 0.046827863468283354, -0.3982572183052814, -0.20597066418919813, 0.6163148741610573, -0.705353275483357, -0.7937251983258027, -0.3653313037630983, -0.46489057476592877, 0.4640945660305239], [0.11901189392434999, 0.3115901414832968, -0.1878275422780344, 0.36625216794435494, -0.08162611556167097, -0.49702631724659285, -0.4297797853881078, 0.3000037404430654, 0.2631981287727995, -0.31973424262517003, 0.4817603582615563, 0.22216395520581822, -0.17029702916127137, -0.6090419277950108, 0.6920844829376884, 0.16251056757960797], [0.413109983330981, -0.4536919156283457, 0.13545105524531798, 0.4359928808719738, 0.10570568214434654, -0.00994283193633719, -0.6011909636692251, 0.6311837916212673, 0.3808641661642381, -0.33361019532745456, -0.8976904403205486, 0.5351931137321065, -0.5343925503274094, -0.14427705208869795, -0.08179742319686277, 1.100735936054084], [0.2195007418087281, 0.847955699379322, 0.17918726257284692, -0.3809368350798996, 0.6017873308642798, 1.0388276039897502, -0.2222334126978703, -0.22407827214732892, 0.5994431340759692, 0.9782951302742695, -0.1658701779458062, -0.3066303498974745, -0.08394097420963777, -0.1484225109675576, -0.33296048910799975, 0.25126196457115085], [-0.06200769756867282, -0.23022505580978783, 0.35354445530911305, -0.8285808793509615, 0.7631296669581825, 0.8410315344276768, 0.3109326677130897, -0.24128508835690846, -0.4555625928666463, -0.41331802310446697, 0.2697602521972687, -0.210149226707409, 0.3994190680464254, -0.046640456932313876, 0.2072623250487357, 0.6699467683449556], [0.17178555194459297, -0.508049664326289, -0.11612868070253997, -1.0525631711889991, 0.12232501353477569, -1.1407949810476354, -0.2685335116918801, -0.4028389018551932, -0.1890039487978473, 0.5311725124930775, -0.11011728155762074, -0.028411087360538954, -0.7906147697347761, 0.045287397585704, 0.37776232360423373, -0.6462765582273472], [-0.33701625550469094, -1.3210772586859334, 0.4743063085703021, 0.10322106420456906, -0.9146169011400178, 0.32184831893230903, -0.333444034889175, -0.5459106628494251, 0.9385778810958754, 0.019404792003799326, -0.46509571766628227, -0.9813095014179929, 0.31006500095326356, -0.18544871825932516, -0.7416961008331533, -0.05340799467144612], [-0.5122148676922755, 0.21283506163487487, 0.4468297377263783, -0.08228462275803353, 0.23017736955589893, 0.277295671158371, 0.7024179945965753, 0.18767466065677352, -0.5983031388085995, 0.6060430352321212, -0.20400417049187916, -0.12767549736941622, 0.12166633228048064, 0.21063768856165072, -0.24583270660182788, -0.10570562502808768]], [[0.06721216525412775, -0.5415691147178021, -0.8567059852847375, -0.016128742529128257, 0.041765212852006114, -0.29813466460756577, -0.35039314409480415, -0.13671681420737727, -0.08293463028426644, -0.21350360343473332, -0.11296731758088187, -0.08511463182514223, -0.46892841092023524, -1.1521761386885851, -0.36237442217963345, 0.017850517424585946], [-1.350372986556934, -0.606877530434861, -1.075825599032539, -0.2902964656593432, 0.1839067749064105, 0.16681971997650485, 0.26170097076015414, 0.7256130468359004, -0.06813130537238667, 0.35930965512523627, 0.4830804997899433, -0.6944352912227342, 0.48445279164199495, 0.3869445573676633, -0.04005530662503655, 0.4451810053137932], [0.3300185174769696, 0.03889375571328617, 0.15974493777228005, 0.41555029742293803, 0.32750668565402835, -0.6922507175079711, 0.42505235163615823, 0.5027304173047702, 0.4024777178690568, -0.0566792159651765, 0.2231429608492576, -0.35206383655655477, -0.04698349805858472, 0.06714709051296197, 0.4212601826878591, -0.5134273620974368], [-0.43016704562430935, -0.18412088067732493, 0.366894260365679, 0.12543813213249258, 0.6006470660802792, 0.36505461743045736, -0.3075531518377529, -0.7094019145191247, -0.7169577321274966, -0.37779334530414754, 0.6315950242012539, 0.010102254809652217, 0.32629982848498634, -0.6754138075077313, -0.38588337204136186, -0.45079418079015476], [-0.420534667099443, -0.021525669620975803, -0.11306552930226423, -0.448084380406757, -0.3821847309402412, -0.06471705160448434, -0.6769300283425977, 0.7025241389450814, -0.23612201412997272, -0.30609449610134143, 0.15428622042145268, 0.5072730547605088, 0.10089538414147982, 0.08781412976163545, 0.3606862615062491, -0.253771842105786], [-0.3358433419093721, -0.22418880172339598, -0.3309033936382875, -0.5694553277112145, 0.8260694579793416, 0.5415998057092554, 0.01074642879449799, 0.5904661470345761, -0.7704939402469039, -0.2333800477695454, 0.7300304177152019, 0.39012409383381397, -0.12404806631220335, -0.7155940713441648, 0.6608023859112848, -0.5708396578330225], [-0.3822790847891491, -0.02483048013424037, -0.010599138485559854, 0.09508084152148764, 0.17861465766997953, 0.20024195717491305, 1.2731354921698295, 0.23135463459765546, 0.008073935529693366, -1.0247093751918324, 0.5007030575962952, -0.25664555798085825, 0.42328283718225657, 0.05768614605150447, 0.47532542561802316, 0.6058095889688765], [0.9320754073700303, -0.0739458517437912, -0.568237147046741, 0.02477794669663264, -0.9628254921781315, -0.16144320232374476, -0.003388548919541176, -1.2437307855562645, 0.5336145275910631, -0.5119080590525014, -0.2070466939791777, 0.1148758648953587, -0.03164088098062952, 0.21184749473766487, -0.0238478246426811, 0.13351321089983503]], [[-0.5224566946652408, 0.40776225717503345, -0.3848339902752479, -0.6241988341874747, -0.12108356840759646, -0.587371050128701, -1.57144048160111, 0.835660936465489, -0.43429818653327457, -0.06586146004766076, 1.0512595711860775, -0.5837825465549243, -0.10495888136306239, 0.20214721766701843, 0.5171921997611378, -0.6408230259792711], [-0.6976872695683457, 0.5160560109634518, 0.06898501424789402, -0.6396423446531744, -0.7863299314929036, 0.4618870983401031, 0.6632187183408025, -0.08806809566983506, -0.09974128371388188, -0.2638224387772769, -1.3241493002599152, 0.6593911162330696, 0.0143306609760655, 0.6310313067696293, -0.32142730536527464, -0.16837710522235966], [-0.0383774371431306, -0.031742041086144944, -0.18870654649948992, 0.6730435646034559, 0.6592371293619166, 0.5572414873455844, -0.2590793155284439, -0.056808506014491396, -0.5028869985804437, 1.3195059968046368, -0.22589257636024287, 0.1263311446708295, -0.43403394626529423, 0.30946473857319057, 0.46519117974755314, -0.4572616754701415], [0.32760996322319513, -0.39368005882554896, -0.25023717759249775, -0.04101120705234088, -0.34714643609495305, 0.5064058604517981, -0.1571031403133636, 0.05229713293747744, 0.27774655312498864, -0.40461531262588435, -0.15192151086949976, -0.10830702359118084, -0.914472707709901, 0.8744307336729351, 0.3655665723955477, 0.22480796864010788], [0.36232515100717294, -0.619936572155928, -0.10199930612607123, 0.6343020077813012, -0.25107166116768276, 0.11627280348373716, 0.19385148979756048, -0.7939653872890666, -0.2902006981795246, 0.28970745242445833, 0.0645912844656939, 1.0770847482058588, 0.22927946028427112, -1.1078933273485754, 0.8095169074162857, -0.7042638532525114], [0.2083075232138275, 0.15890733130097023, 0.14522014498107386, 0.11115263902302866, -0.42795918500884994, 0.34726357526232576, 0.12778923551619448, -0.23200695446967182, 0.35831679800724087, 1.204444445666591, -0.5864401217367907, -0.30221794688850656, -0.06095098203247945, -0.48701382539302085, -0.019070040450722976, -0.47864139188176863], [0.01767642880163884, 0.4869934892270112, 0.12769828473402178, -0.15588540458634004, 0.22001999345689952, 0.11418954432814683, 0.3605179480397751, -0.07059751503614768, -0.6990570042707464, 0.3674935405683134, -0.17232190137406111, -0.14052715132719676, 0.45122676385421845, 0.17707129561547968, -0.21976625041545625, 1.199189191295347], [0.8651802017145515, -0.23195863712442788, -0.8796930012629688, 0.5449666154196016, 0.2156410963733703, 0.0037543761482933874, 0.16160690531750152, -0.05703822864360818, 0.32054082232208675, 0.06168648476988013, 0.4852220031621206, 0.5661964819441057, -0.012129910836361956, 0.24710887692708225, 0.3018607214569258, 0.010753048087983712]], [[0.3256792825168032, -0.0377480117265593, 0.48821114920928527, -0.26910343400314846, -0.0915281649727214, -0.823029346863591, -0.7175820729936371, 0.18316202346904464, -0.0468605655982758, 0.01371297739519109, 0.7592737012890176, 0.017704394260421346, 1.152666441028215, -0.8833670975557214, 0.0879126255826997, 0.6105322061684435], [0.2155292803332338, 0.336975295430166, -0.2962066207314211, -0.8399605847058955, -0.050152532161479586, 0.43811778570245746, 0.21194920430785144, -0.5221235301242976, 0.3626888209587309, -0.04181238461308555, -0.0625235773671887, 0.2518687143731491, -0.47593168004032954, 0.55175125541813, 1.0012286037841214, -0.5910904996771824], [0.07000110609713613, -0.6030631288583067, 0.15515412263131603, 0.3203300367628304, 0.8801120325333524, -0.43668153590643505, -0.23961387724971964, 0.35933387326230204, -0.0446833069373824, 0.24341021500447751, 0.8830041065856171, 0.37114566610693195, -0.6152304301026972, 0.8735286910759761, -0.6519095751801138, 0.4026341092250634], [0.08244281996402493, 0.24872207030184318, 0.06940645366945751, 0.23452857992926157, -0.9751340059922143, 0.17696200686691962, -0.44271894908315296, -0.4091554812056432, 0.016539364604058, 0.04930082779337359, -0.1752083889597882, 0.9218194700658097, 0.4005812075108889, -1.0025817482905581, -0.2626330967035112, -0.5292720794104543], [-0.3158889029906293, 0.88185102572176, 0.020641702865160875, 0.4457621250250069, 0.3287345389079512, -0.6354836650320952, -0.5494488409469152, -0.11613527298609154, 0.1319691984075074, -0.4068120315418191, 0.20496449232141387, 0.020539992481830752, 0.1580318176993738, 0.6405262001010041, 0.3242213911862815, 0.9298395216783669], [-0.20361335074373005, 0.06623727843229782, -1.0086772564406872, -0.07693539874088058, 0.431215030783547, -0.810520914565171, -0.45554864546732327, 0.30617998404310126, -0.38025737732232306, 0.30242353368252023, 0.9161173084220243, 0.5543807333354912, -0.6908924384426579, -0.07066878260352331, 0.1251932438272617, -0.2197387280456432], [-0.07386073625843835, -0.08560612015206696, -0.9495056169689565, 0.3446341091077204, 0.4291359582004693, 0.7337510754941484, -0.901785455094244, 0.15812463403996288, 0.4959686081353069, 0.031203973348971528, -0.4700514879695308, -0.719443721714706, 0.5983724488477363, -0.6303619567067451, -0.5104998539161664, 0.824508023843017], [-0.88060987255516, 0.30037068504774267, 0.20679783699644885, 0.016722392751502144, 0.20574529674387987, -0.7032105822235688, -0.00539080953881561, 0.40912144852439514, 0.17570636159742503, -0.02199102516268719, -0.7885859724531523, -0.20961584948451992, 0.7384582001430235, 0.5169525147438104, 0.29015671956369954, 0.07418408689240855]]], "dtype": "float64"}, {"__nd__": [[[0.4778450522199636, 0.2722378609191573, -0.46924702398270873, 0.28780482698607956, 0.3922045676195213, -0.6289743030628431, 0.5070885207659869, -0.4416811098127255], [0.432556982719768, 0.007867287750105244, 0.1535469711223965, -0.590584542261699, 0.508031216783443, -0.09952505944615847, -0.23499455098608607, 0.5998906569556413], [-0.2750699156576959, 0.12331712376237644, 0.768640205313334, -0.28388269853379583, -0.5104612468907068, 0.038005291023508234, -0.25673522408642346, -0.10299701522427702], [-0.3868110688072021, -0.38683244486555196, -1.0419878191057819, -0.7057833072704075, 0.07977506555078458, 0.023296884457049104, 0.22843711909517966, 0.6041979634418176], [-0.459436876315615, -0.03148982501360854, -0.2835357598923303, 0.2468819422958467, -0.38484469537988053, 0.27146580975145124, -0.62606937018271, 0.11524544427941653], [0.14591873971110197, 0.5694582092737472, 0.27758718002607385, 0.458321806060364, 0.12394823023721353, -0.3378265051398928, -0.42364973998182126, 0.10342871633265209], [-0.3583535918497623, -0.5343849297570553, -1.075827044241486, -0.5030501297420329, -0.49777440724324784, -0.7423890590955063, 0.38859927323961024, -0.8115876338193448], [0.05244845658836979, 0.08251670100424599, -0.2868344498448438, -0.41774368167071774, -0.15333665698188667, 0.41188462824396843, 0.3793437875972976, -0.18414457639700182], [-0.13754128419456108, 0.6065185566284378, -0.4633540508603484, 0.017751381820637204, 0.43935935163365475, 0.6155110474811184, -0.5021474610552402, 0.23503760094127604], [0.016722680306954507, -0.8644842473040708, -0.4682120020233856, 0.29552233751375057, -0.014488588310444148, 0.007562024153861674, -0.5819399331206564, -0.11898809973622697], [0.29872167929533466, 0.49713041189311724, 0.29305114413769945, 0.17107076263645773, -0.2670485546310278, 1.2303380658597642, 0.40585455361451317, -0.9780970476890292], [-0.44787667983311485, 0.0933151724115507, -0.501096585867589, 0.508059971916082, 0.1470904272062598, 0.1519368446901579, -0.17885560019283328, 0.9149802409646353], [0.05645273958547753, -0.4617581507948492, -0.8174380523583262, 0.15718705580462486, -0.10894886053895834, -0.5675537297891071, 0.18700975966401473, -0.38503087386146706], [-0.526225036363707, -0.04743161346481004, -0.5390247517232379, -0.015864757198700962, -0.6208086325715135, -0.4012061117376422, 1.0736849830319215, -0.45454205045932267], [0.07797827702576424, 1.2977228303094361, -0.022395753974609946, -0.17012441721562882, -0.7610748164873343, 0.3707032035307585, -1.1458090012242337, -0.7284644011369509], [-0.09538839487031542, 0.7510436546912866, 0.9555729723004007, 0.5807293279553758, 0.08772850958993315, -0.2993654585171781, 0.21117601287691462, 0.652968141020886]], [[-0.11078701740611181, 0.5305946057202131, -0.6455814580943436, -0.0012419372364282446, -0.1690821831865176, 0.19878283578326453, -0.23960327950002622, -0.32274161221908304], [-0.033247483783702354, -0.22776456389790425, -0.49262746062221613, -1.1426731555787581, 0.8062736843084314, -0.9453968196152848, -0.1691107526243196, -0.022071812228252136], [0.6995850509375678, -0.1622571917639994, 0.038692695648098886, 0.011766629507523592, 0.27306263870215675, 0.5949733899167489, -0.3542168703023083, -0.07019474591542403], [-0.8490245891025195, -0.804968577354121, -0.6651452068251181, -0.8283245522927297, 0.31486193471111473, 0.036041006276541124, -0.4693286070714989, 1.1326569176862429], [0.0987501971951765, 0.5222202401945247, 0.27031118591399594, -0.23857764522088162, -0.2181818653101869, 0.3153805845622644, 0.049072858540515546, 0.8563448570462908], [0.3708202883843402, 0.3116222025501555, -0.3628605522834577, -0.4753933555311092, -0.4182961374594505, 1.0435619192357666, -0.5045635849491674, -0.562602734801637], [0.3192732931220161, -0.32069785938978074, -0.060918993329136556, -1.1606101959153041, -0.016965701985301312, -0.023962561835335013, -0.2753533031580627, -0.18527424599603481], [-0.0631541765966047, 0.24327206396847761, -0.30161449916401634, 0.3592682462705801, 0.45941690624580306, -0.30456864711346665, -0.014332381875007015, 0.24907487015404037], [-0.9829482559665859, -0.2818996546708086, 0.9729417688403819, -0.674088181078744, -0.28724745179210626, 0.19483530074045285, 0.1893796802036249, 0.05064039369823254], [-0.15124058932091253, -0.11690399070294734, 0.10732161274061007, 0.38469756251227794, -0.2515660395429031, -0.6682804324148035, -0.14062490618873394, 0.9624842601748308], [0.1580695179590988, 0.03769566316182594, -0.46828929663435104, -0.6159879892239178, 0.11169246832397721, 0.10865791930839203, 0.14326211897020294, -0.4515057970931822], [-0.21161446433883604, -0.1816829088796929, -0.3148357151475898, -0.3233281367064671, -0.29326821120569113, -0.8566038257799183, -0.35060956514506286, -0.9080831811379556], [-0.2850041222514821, 0.7034741589853828, -0.6528294405752474, 0.48940976818284015, -1.057450531062407, 0.188961171514875, -0.8135579905090615, -0.4751802314805204], [0.05335557455226065, 0.19020082899200283, -0.2627116310801071, 0.7329018058954782, -1.082746585882849, -0.022847687677520667, 0.29776579441369005, -0.12098227825570684], [0.19880630102582827, 0.0072360617158525676, 0.3163820363734972, -0.017922918728428643, 0.4983070259742097, 0.23250820541795555, -0.14456216015586504, -0.06432595103807469], [0.0008038674538043979, -0.04542310658239972, 1.1233407622773801, -0.2929744961456516, -0.11147794788381793, 0.3614591152511999, 0.39311119457240107, 0.4183942835240441]], [[0.10375490838367671, 0.3119383879908935, 0.9523993584141863, 0.5525521072390331, -0.06445616767827199, 0.5254063015981699, 0.5949691002091559, 0.8788177821215293], [0.5513495712361844, -0.36995188663125494, 0.255782998211883, 0.01474802713334004, 0.011265976673898217, -0.0987531991976831, 0.09162937308969048, 0.2856053688977729], [-0.026848396688807705, 0.19018455685826904, -0.49856454466598715, -0.24889090150964702, -0.28575946750297465, 0.2578539313479279, 0.27755449318774605, -0.2833532974125566], [0.09901655601351697, 0.05235479692191612, -1.0511128242515293, -0.43489892061517005, 0.21511837173288126, 0.4888899561033966, -0.13122753108007604, 0.12782603678130522], [0.6553325717786472, 0.2883792023136689, -0.009667700617927528, 0.5758233264315074, 0.030518218147381176, 0.6320688601877444, 0.22493414743888537, -0.13613835130945134], [0.07218696631590733, 0.22941927352617292, -0.33573051248351404, -0.2490514285161096, 0.19133147268913056, 0.08384887502647141, -1.073350252301454, 1.5168777247232827], [0.12356023635820766, 0.31666075922296355, 0.6602471078572205, 0.6783928514698166, 0.22277328928598109, 0.9654446755203642, -0.42260309879750985, -0.1592938494061468], [-0.48377749658697367, 0.6875299002567168, -0.2972654225616493, -0.10302685469103852, -0.1006976477361591, 1.74477175055289, -0.6013061079116866, 0.7276912094101418], [0.4195149602677833, -0.2906163663901741, 0.24934614412552014, 0.15778844805864756, 0.08880898590132198, -0.6017963606875025, 0.3727666507538846, -0.07611499549632823], [-0.4191826664694975, 0.5793309181948906, 0.3429320736149103, 0.4566131551899832, 0.2773592134181084, -0.7378828872733508, -0.2398291614934987, -0.07830847920578608], [-0.5106068516385445, -0.7242978193533465, -0.3100035005691983, 0.6254468047080921, -0.5921676108022724, 0.10889260525947264, 0.6653197629673053, 0.04806811075853909], [-0.15183659016640733, -0.5412745397868131, -0.31028199911890103, -0.03572082240196448, -1.012182477570636, 0.06152470663106938, 1.0187284579505413, -0.5156607301445166], [0.7464110688038964, -0.26655892712169593, 0.5188749900864067, 0.6508987375145135, 0.055538297799625984, -0.42997913127458476, -0.18348001786990834, -0.4195417912909272], [-0.2069105563981606, 0.5731055269337363, -0.48369899699338853, 0.33495848725244476, -0.2691306784114582, -0.5104806684434342, -0.5386897233360486, 0.602772642925366], [0.4743309612348988, -0.11949262485539015, -0.23929779051196548, -1.0440548660280624, -0.6772898137378697, -0.24412025083621372, 0.13688665249618676, -0.5618688645625834], [-0.17958944810681632, -0.49318837556925715, -0.13283175048901472, 0.3177882852521342, 0.6412014808587502, -0.4336860918751548, 0.13046431187060198, 0.08697404568758135]], [[1.017810606848748, -0.11176876377818344, 0.34540997057484374, -0.3250424265747807, -0.48793066495765547, -0.06455922239655541, -0.20309212833912735, -0.10019500613870694], [0.33086555326056755, 0.3352239577583559, -0.3397402118282543, -0.34548307587910254, -0.4269584924177069, 0.32280756519687653, 0.11245521321634239, 0.04507476881075782], [-0.42921790325672304, 0.0914901886873759, -0.28784896743275296, 0.2359067417636358, -0.31997635590669526, -0.0236279167012781, -0.03529023745579673, 0.6461974176490498], [0.43324348015317876, 0.3295762354381062, 0.41230125582784366, 1.228755200863789, 0.00422231766215199, 0.18198071621240888, -0.18320597909315642, 0.2237388907904471], [-0.3705297968495236, 0.37253288734272605, -0.7969671265267515, 0.45519352224517773, -1.4771083851735822, 0.280032575481779, -0.7742392143289348, -0.9982328896671943], [0.3512183291772523, 0.5559808220999393, 0.4222297763045233, -0.37917645824326585, 0.7523656341374322, -0.0005257695685147867, 0.15656930990510184, -1.3625404988885228], [-0.09595088288876806, 0.1988545548352912, -0.1623378030588927, 0.4048633909974019, 0.49324881613634375, 0.8916772409191752, -0.45291829464262245, -0.278077084678989], [0.14733272614791781, 0.528191321645909, -0.9469828068618376, 0.1643835398318159, -0.003189024619467691, 0.5812468764581857, -0.471708662309031, 0.41774459229235694], [-0.12136689155578792, 0.8325973099427776, 0.14451366152028952, 0.2658161694086344, 0.2362926855425134, -0.34249644390625467, -0.3415229955388554, -0.32899869256806846], [-1.1912753836837877, -0.9538686463327098, -0.18843244283675234, 0.7812399842489356, 0.07006896533586969, -0.3306021109765527, 0.3627705076616068, 0.27140192716396594], [-0.48600655846077423, -0.21062724701237281, -0.8109246082678269, -0.6499717070215513, -0.18405248586942047, 0.587471375629809, 0.29276993713783056, 0.7374133247963616], [-0.1878526388774244, 0.43471163310154504, -0.6290484869553826, 0.2381710157986849, -0.6742529873583519, 0.4840706146825077, -0.5813055677089236, 0.5044654071829906], [-0.47342438535823544, 0.009822708027024788, -0.17213387192912238, -0.644434209905697, 0.548190332235054, 0.3969247016411825, -0.06112467286255312, -0.026628398188770087], [0.005188836146824646, 0.3805802714250458, 0.11582050237280692, 0.019364074019727975, -0.07904339888825014, 0.661182666066263, 0.5914512079801072, -0.013527113227369384], [0.4957388515819182, -0.7176253995579804, -0.5324314551668635, 0.2208058159106856, -0.41782971927262647, -0.5643595584969534, -0.009379681266237795, -0.037535175778993096], [0.2786958529559457, 0.3702499038328964, -0.23176085778578243, 0.3125613404292672, -0.374738543943409, -0.29840145774220106, -1.084126447938614, 1.0471949258165791]], [[0.2174382269822008, -0.6484750311445542, 0.5213997593323672, -0.29703175588519737, 0.02740027243030781, -0.20493906948397117, -0.6006284036979475, 0.23674629222365437], [-0.04510433307759897, -0.6278364726064585, -0.211377453817642, 0.037670916169266315, 0.4249111971451562, 0.009160518185588397, 0.0859884049001075, 0.2618527970251025], [-0.33156369114000456, 0.312687142867619, -0.3997909338766077, 0.8025617158428706, -0.16181077767233956, 0.08892166302547416, 0.23174645818511697, -0.7666614392575763], [-0.38522417613122245, 0.5615449307839798, 0.5801995156168434, -0.7959627025231727, -0.5140546868959593, 0.46146377289511925, 0.6018955235811001, -0.5579836243538812], [-0.23359153925764037, -0.27612048559084634, 0.24467128028314344, -1.0590917937346365, -0.21988641169578205, -1.0165199110543657, -0.3185946102302888, 0.33650431589140717], [-0.35226573571406017, 0.20064774909650138, 1.1230923296500186, -0.7872177557669324, 0.20594785999485904, 0.7972580112471331, -0.09881213722362295, -0.480416022700352], [-0.5986187072507142, 0.06417599148222537, 0.10304169573690787, -0.025223114177373485, 0.025409037464720192, -0.18693820127204913, 0.8514567758183847, -0.8771975308991314], [0.968725215653515, 0.6823106113014682, 0.05941661665373604, 0.45306587083360134, 0.7234100627945474, -0.2537506908848756, -0.775553558310157, -0.42637770816895], [-0.025919834864101217, 0.8990360702727819, -0.40710067082813484, -0.30485736377184464, -0.7566688767896322, -0.05850258371245659, 0.39489269618397005, -0.21121348414416097], [-0.8532078123202329, -0.07802189852892472, 0.1524908374301123, 0.4044120291457457, -1.2575511176614314, -0.4900919718887906, 0.2943229809736679, 0.16122991440863108], [0.7324461567461205, -0.11521219232616442, 0.021292015681164702, 0.08017778087862122, 0.1368563577909256, 0.45352683764924945, -0.032420711436199366, -0.13006797472868614], [0.4765768891613909, 0.49865696290894623, 0.4857031379013488, 0.2683323619821037, -0.2508179058058939, 0.46022506839674343, 0.9901068577357622, 1.1890583529811372], [-0.00822713963090688, -0.09479681626067366, -0.19447698209485167, 0.2929401589656453, 0.26324818349539597, 0.4623323379812155, 0.21160922548445213, 0.011877286383687192], [0.6390450427440082, -0.17551947535760407, -0.314731986045044, 0.11298145278177062, 0.28715648818641637, 0.40671965927380993, 0.7935104773691176, 0.2498159104531438], [-0.4487963473632254, 0.041107636556518376, 0.9182869086373784, -0.45986289063519115, -0.20697372268514863, 0.4646541206853792, -1.7297875398925864, 0.1992999456069664], [0.42090232669379474, 0.502369152743784, 0.20148029421122565, 0.2599840961388458, -0.22943680464365726, -0.02122809577520364, 0.03754854212359939, 0.5072171900053823]], [[0.2706550539069541, -0.08830772853754276, 0.43498619324565985, -0.2210385298143947, -0.7872209822502817, -0.09413069992242735, 0.6158160770510904, 0.3172697123739907], [-0.12855736051424493, 0.557930655599369, 0.37324801696620286, 0.1203366043874063, 0.2285330245218847, -0.010016338671881807, 0.20086619784095136, -0.6232918094103446], [-0.15782157468749033, -0.3260228928255413, 0.2613445711292692, -0.037602618842142846, -0.6396905692107971, 0.25066653022774754, -0.004175556301513328, 0.07789339240296324], [-0.34895605926411044, -0.29204469614673984, 1.0606560234881435, -0.7093535725347757, 0.08444495720808647, -0.43743217943787877, 0.877056976059732, 0.8229805727959263], [1.1913275503889513, 0.850760740627949, -0.7072893759722791, -0.4185319637092256, 0.054407910521560014, -0.17373660269536417, -0.7594248505740633, -0.13302463779164522], [-0.05924989437671154, -0.27122236537377276, 0.4677920962923433, 0.12404452547935615, 0.12324971367074933, -0.4623709365584628, 0.2952038700831805, -0.6794636359467694], [-0.2760790223447148, -0.498981735488376, -0.2426996943929492, -0.3270262348185756, 0.028437134762754238, -0.17226879761006383, -0.12826796915965985, -0.21493347616161831], [0.4636944217000861, 0.19535147386258772, 0.46183919020240266, -0.24564585312448453, 0.17229450317229852, -0.545220830133533, 1.044242873555209, 0.11484860654803947], [0.02893803968167608, 0.23991128849201449, -0.5726420408188352, 0.3374230223094335, -0.4290002558863579, 0.5409356261123985, 0.6689965046077904, 0.6858451336479029], [-0.5747687121359084, 0.944241082720049, -0.6211186515990756, -0.16275210085273673, -0.36960469174069693, -1.010229910983303, -0.47659099173692215, 0.19577102570025398], [0.08736300919564302, -0.08647976052593875, -0.22629577522324346, -0.42795240298512094, 0.7572701903555108, -0.061412436577901554, -0.4813496825750314, -0.10118061189415081], [0.8627639471143619, -0.3080935039016386, -0.7866677357719802, 0.9323771500790828, -0.23140472278797183, 0.1669004064795324, 0.5777106887017069, -0.2965980160974506], [0.35130323976991235, 1.2868590461955607, 0.4947071951827008, -0.16109504071261113, 0.6234817536070436, 1.0051376142715136, -0.6555917787783353, 0.1254401804417977], [0.5137215596180552, -0.19386195684284277, -0.11818996032500038, 0.09016149361994216, 0.4621690140684704, -0.1462891362755391, -0.4468868690948133, -1.2831716174798755], [-0.02029510274284788, 0.5738921582024105, -0.38324759032908445, 0.810245601855174, -0.584751034935932, -0.45267825208394485, 0.12516508252391878, -0.14599125935260746], [0.6301273256340686, -0.35085737313622545, -0.5288791663870179, -0.7606498555156276, 0.5355236038772275, 0.8870126506605444, -0.233862696765151, 0.4053434722442635]], [[0.10279138335025308, 0.8161987963790688, 0.2511740980056305, 0.09131551710556263, -1.0668374878349935, 0.3085786537877445, -0.9787621131022124, -0.38918877328418555], [0.06110177002816668, 0.48126895973295186, -0.09177997660348122, 0.7516104933281186, 0.19352048067138788, 0.03460966832210567, 0.14300584864072471, 0.5121643731953651], [0.32872115222529313, 0.2760057207155828, -0.07439787554339229, -0.6596117621347514, -0.2960902252330274, 0.14207666086997198, -0.4002473988573842, -0.23399547681823785], [-0.08146192625051239, 0.8899457953292716, 0.07676760528001218, 0.6021772185556604, 0.11519369915488524, 0.1731666053303414, -0.529689477462207, 0.8089144496883492], [-0.6627272002874904, -0.12142732843426833, -0.6603483077802503, -0.04687791655556029, 0.06936426875120379, 0.1045740965254773, 1.033797746691781, 0.5930150486139517], [0.3292625549977214, -0.4414103589747434, -0.8963156005129261, -0.183922322460693, 0.6177838540507907, -0.001534947923396901, 0.07360699406623158, -0.19953737851400744], [-0.16242949390057054, -0.4565856405685764, 0.26029206065519983, 1.0232103460326494, 0.07033580867546638, -0.1674204916862227, 0.11052773538284091, 0.8074089381506725], [-0.19860462244386415, 0.5090804139344337, 1.0369060088156787, -0.24356180159094792, 0.13243460186232706, 0.38165937130245986, 0.5213796025216797, 0.7236612415933815], [-0.26525313873017714, 0.7340213377238923, -0.6913905904050858, -0.6578254867029868, 0.7199684443162108, 0.3579605928804912, 0.08702498320108003, -0.954788084974431], [-0.4851793203345127, 0.26130244014573023, 0.21028415362590616, 0.38298157805551075, 0.10054955997332357, 0.46643336381971967, 0.2868148620924756, 0.6405745351537264], [-0.36118937489060426, 0.898875861794948, -0.6701257620044901, -0.29512523185404177, -0.5056813735650775, 0.03464965713820747, 0.27978285142123765, -0.23691332412871255], [0.7374264683095167, -0.8606240565659824, 1.1512059491798936, 0.04587011935890432, 0.6430665858533414, -0.08917456609072219, -0.3877451724217775, 0.3979047149577785], [-0.13278918038669604, -0.3996171704661623, 0.4608112847897632, 0.8981531144044659, -0.10526733722199104, 1.3413887937357907, -0.3520953739335905, -0.9362480590024753], [-0.7723046094342768, 1.4751898763686333, -0.26353341266576524, 0.29544119522767315, -1.2266287455848102, 0.5199800499524411, -0.23903026886609272, -0.4749169953753999], [0.15846626820424015, -1.3712266084281082, -0.0003223242601718561, 0.1189456602894621, 0.03799337608331916, -0.15772460937966423, -1.1426506309892857, -0.10873601420460789], [0.39718261778528896, -0.20423471782753072, 0.2740421709660656, -0.6373422194739281, -0.07640239091378258, -0.17529989931931714, 0.18109087656183262, 0.2282060951796201]], [[0.45692525940300244, 0.3910724680360143, 0.42911345542309803, 1.0314923413784003, 0.004500604989152846, -0.10430163250120142, 0.420319964389377, 1.0140748669943223], [0.375271981487859, -0.44500058690761046, 0.5146552309220489, 1.1543717675846108, 0.13268813240121885, -1.021615685705021, -0.8146555844659877, -0.7764843968763473], [-0.04965095035930676, -0.38262224610066214, 0.08392723280600926, 0.43926123954767154, -0.7112118387101557, -1.0277273868012227, 0.28057769273253425, 0.49130744335128956], [-0.0889876452657333, 0.07709379450079062, -0.6592972769232687, -0.2389515880538738, -0.34600370899039906, 0.03144891050380024, -0.7008138058634031, -0.5964383931888553], [-0.211261857205996, -0.3549737071129795, 0.4992934926327884, -0.6379572378467082, -0.8534228247363296, -0.22440078222398815, 0.16958759415507546, -0.24568210104203944], [-0.7423975032097403, 0.40609136686285036, 0.13452439480973952, 1.2112909502410094, 0.6780533269961831, -0.17388077998450485, 0.2685082672631695, 0.5920641678616665], [-0.5438929429423423, 0.34048214196317633, 0.15953646441578817, 0.07527269667484866, 0.7056144081339794, -0.19204069750405486, -0.32081988374016024, -0.06768425314551825], [0.18345353461106315, -0.14397292464739367, -0.09913431897665245, 0.7019989907266294, 0.4136583375226741, 0.43351036742342336, -0.44350530736916793, -0.22133156626191294], [0.544549252314307, 0.29607166310662797, -0.3260587817217274, -0.32777734467386177, 0.33666400262902973, -0.05715430645657072, 0.08317101512322113, -0.4228053962859483], [-0.524502897301188, 0.6784440659210431, 0.01589863401835296, -0.36406052137611333, -0.12286150512694455, -0.6140634114218861, 0.31550043624468815, -0.23733681218488295], [0.4912044410467047, -0.13816213276480024, -0.08785273790472929, -0.6009454772826744, 0.6546307005468477, -0.03803493392673907, 0.7629076944082885, 0.11186765107154929], [-0.029487626720452177, 0.13325410680224536, -0.3678561585251019, 0.6298606448972494, -0.16254220876539496, 0.04866639359117055, -0.13158496939899642, 0.24266576538931556], [0.7343564590362633, -0.1247220632019626, 0.3759029250972424, 0.47536982616745116, 0.08515614692211683, 0.16985491933040503, -0.6106349806745363, -0.15805538803784502], [0.48582027287405205, -0.3997728673399232, 0.00669871144198331, -1.30648570997307, 0.12636522309795975, 0.16822760416258228, 0.18205185211402178, -0.22596678682877086], [1.1025494726295828, 0.5531703364490722, 0.4806443044771014, 0.0080241525359569, -0.8813407235600038, -0.05946259538596234, -0.10103298870750767, 0.4282302586663275], [-0.3634549981062997, 0.7388141518407736, -0.027056494190103416, -0.4000459950024045, 0.34428014643279115, 0.8976482860642913, -0.4348768198376095, 0.11070787458659431]]], "dtype": "float64"}, 6], "e": {"__tup__": [{"__nd__": [[1.0960259254928706, -0.6459682031939606, -1.4623121257388878, -0.3382481692547532, -0.7667855986380192, 0.6683768256405852, 2.072572967896999, -0.23390430482636304], [-0.5287222121191886, 0.7172662882942813, -0.44820499817445614, -1.6476697417821535, -0.052420158866202216, 0.0965702656625752, -0.8249225702648268, 0.8727633477849386], [-0.22738951372671173, 2.244329401922068, -1.672808278100535, 0.5831850960247085, -1.4196410925633292, 0.38742646526980223, -0.5756869711711791, -2.8212697695398097], [1.4606694445675625, -1.4055894021976076, 4.5137276732105445, -0.013192270013071772, 1.0268109652234534, 0.20647503763571373, -4.936351804855619, 1.9302696293837136], [-1.2741054340552038, -0.6971100344032614, -0.38480144781572423, -0.9422016370132764, -1.7874371136515592, 0.020839461678363813, -0.0822511404706906, -0.12803738134753462], [0.3407796229984079, 0.3978660717292903, -2.881340693815427, 1.1776567794712558, -0.20345429169511048, -3.532906720330786, 0.8324509213535898, -1.884791864493667], [1.276648820416328, -0.6085363455906299, 0.4092150124232149, 1.8796144042819922, -1.8435524920963093, 0.9907264662377775, -0.48361910069880026, 0.06455189657078432], [1.5916534937215228, 3.1744973741449956, -1.0408228236289672, 3.6340809599061172, 0.12115141334148632, 1.8689513641416846, -2.540852326967471, 0.29354417770043084], [0.304197290387925, 0.539674936828934, -0.9936274609309375, -1.3607299345310633, -0.9927781984747448, 0.609304741210251, -0.3803252475751643, 0.39420693772039594], [-1.1408196823643117, 2.3217811755330002, 0.8025680397381574, 2.951481373871544, 1.1112916655919731, 0.09273172517913608, 1.021719738507079, 2.4508985801361014], [-1.1260883215461241, 2.1423151204125275, -1.5452539378823726, -0.21509889509454755, -1.5255743461779898, 1.1146989834952616, -0.13099400005466622, 0.3745160800025957], [1.288147759245697, -0.12076730922944351, 0.9251436065987209, -1.2293455085404177, 0.3778280176134084, 0.5152900280189832, -0.18049407087808408, 0.15521313898259487], [1.2553106378921788, 1.3708806645390748, -1.7158036361609033, -0.9016417363173947, -1.968489121522016, 3.721418906142877, -0.8957635951291921, 1.0386076186874715], [0.15565589571090965, 1.8705866316174402, 1.0906027214062786, 1.7523566736990932, 0.21533726962318234, -1.195978341656349, -0.7365480374599507, 0.6018549562457556], [0.30312683557093634, 1.562665437517619, -1.7963663369603045, 1.0577801998303402, 1.0415022083913301, 1.4686771363887174, -1.2353589987667608, 0.2201816635654953], [-0.715027099387804, -0.17268507631242605, -0.5988131723484053, 0.5340181304277007, 0.17692568895583677, -0.23258714646164558, 0.050215366481482095, -0.3260463162597086], [-0.12975346656406478, -0.07834353733574037, 0.05357922993002318, 0.12976465314795677, -0.1169526477454765, -0.07206672037831613, 0.025633593265778735, -0.18536506203181424], [-0.5343825629926895, 0.43329389816985464, 0.4563407261101154, 0.12838147862535756, -0.4641381548601081, 0.29113460443453315, 0.372608591373752, -0.5158583251412504], [-0.19186136690334588, 0.9698413469115569, -0.6184440754391547, 0.25966501865344976, -0.14178343768975876, 0.36864848254605664, -0.5099649492345267, 1.1071080730245255], [0.5597308441820902, 0.6617516506055421, 1.0971682653993962, -0.139585338867637, -0.0362993725444057, 0.756033789872498, -0.8876990257895673, 0.1922648843482344]], "dtype": "float64"}, {"__nd__": [[true, true], [true, true], [true, true], [true, true], [true, true], [true, true], [true, true], [true, true], [true, true], [true, true], [true, true], [true, true], [true, false], [true, true], [true, true], [true, false], [true, false], [true, false], [true, false], [true, false]], "dtype": "bool"}]}}, {"a": [{"__nd__": [[-1.4075519778872658, 1.5730412782554066, 0.13058077974126012, 1.1138342924182898, 0.0890552624359499], [0.5599504463126126, -0.09321739531958002, 0.2739653956169758, 0.6193220217473235, 0.4431208758617539], [-0.6821291216650311, 1.3066645378912922, 1.3942002087358556, 2.131145800915433, 0.16659663961681828], [-0.07815947748197535, -1.1100421412420924, 0.06171430557412598, -0.10957079775542604, -0.7724420623504011], [0.9218355050072282, -0.46994129253463546, -0.3405862142223779, 0.0165616174740972, -0.2509470411456719], [1.2613597286864815, -0.0930913961123976, -0.6657054408551314, 0.6489136334450891, 0.38768483696900724], [2.8069537219915297, 0.9671762415642912, -0.35233384782174487, 1.4466296605709106, 1.0620593966577447], [-0.7148979073655825, 0.49996658666308097, 0.7376247525647186, -1.2224233549269716, -0.22350585660312172], [2.5634641754418466, -0.3019250656365829, -0.7339384082952753, 0.7574535590112766, -0.9420870352014296]], "dtype": "float64"}, {"__nd__": [[1, 2, 0], [5, 1, 2], [1, 3, 5], [4, 0, 1], [3, 1, 5], [0, 3, 1], [1, 0, 5], [3, 4, 2], [0, 5, 1]], "dtype": "int64"}, {"__nd__": [[0.16729084834222677, 0.26335402878389624, 0.569355122873877], [0.2776153057093963, 0.2839858034439615, 0.4383988908466421], [0.12521204023350388, 0.47824130908479867, 0.3965466506816975], [0.40902421778842324, 0.4907712026388622, 0.10020457957271468], [0.1949001207666464, 0.5336259219235095, 0.2714739573098441], [0.4186850524336621, 0.21362663901252918, 0.36768830855380874], [0.1756355697564786, 0.5081417637542097, 0.31622266648931163], [0.46291720375432144, 0.22960104238642057, 0.3074817538592581], [0.1630835900097232, 0.40452094250116144, 0.4323954674891154]], "dtype": "float64"}, {"__nd__": [[[0.01950177988515022, 0.6463124800496002, 0.43687496656090474, -1.4884513854821222, -0.4397652288436266, -0.4453985621488005, -0.28056362769867604], [-0.9067179643162783, -0.2561420717937318, -0.4303195526956556, 0.5322352089769957, 0.40239398475298965, -0.6282017839943501, -0.5749951281355278], [-0.27780716556606316, 0.020020316117574915, -0.4972558590418957, 1.0673623395581435, -0.3037853737551555, 0.6455521126564943, 0.07075100578844819], [-0.07003841789930915, 0.35700909394544705, 0.6120081065047093, -0.12763736538959475, -1.029703374644377, -0.4261095855941221, -0.33793518137929174], [-0.05649829964542345, -0.033802084707324744, 0.4315454501640525, -0.3531702713131078, -0.6452581078834176, 0.15770807521508476, -0.4904218597022958]], [[-0.5833881167894371, -0.021691840901808646, -0.5783360879699102, 0.8123445373848475, -0.1750190602027017, 0.40508431287781343, 0.01776130661445504], [-0.2938408993800093, -1.0225771847031895, 0.4318243883301217, -0.6636380876732454, 0.36998907797381897, -0.27540243962107447, -0.07517436745091446], [-0.9546782892234962, 0.18648455128919694, -0.6319731932741015, 0.1454543819253365, 0.505330177633438, 0.1072722559209677, -0.7220038171034749], [-0.6012779950357728, -0.01675009979333043, -0.4752352360250878, -0.016014170882316554, 0.7805838355808105, -0.261486933663552, -0.05403019072699479], [0.0443583730434567, -0.23610687602110486, 0.5841209687873018, -0.4436910475255051, 0.6196049116609145, -0.44995476682935875, -0.46782020514474343]], [[-1.072780658942267, 0.20248412138135916, -0.023187028594051282, 1.0180017700574626, 0.28540676949403077, -0.5080911900378953, 0.7347332510617481], [-0.9708740084081153, 0.11885481254214239, 0.030776362659192688, 0.01656933941667552, 0.34446823106125657, 0.22306953413249428, 0.3275730287643819], [-0.01749901324055876, 0.5370153536628067, 0.42109468402364203, 0.4119221075309376, 0.26274362405962165, 0.25657118470888435, -0.012558183144546533], [0.49744460918261973, 0.2625292805656092, -0.2897138665428889, -0.3355985185285126, 0.4294297083006146, -0.4925610115359621, -0.16882920785142902], [0.5360818242585705, -0.24167990441109283, 0.08838318961853205, -0.7259123608878665, -0.5873115000138157, -0.05669907616372401, 0.36045566764774306]], [[0.1571522853971231, -0.5744478009600051, 0.17225042536573598, 0.6772368548642806, 0.5636483683230609, 0.4615472748677422, 0.32420154518851735], [0.6371919620179681, 0.09344315495595393, -0.6639809961929818, 0.3271343165016762, 0.48010320317167315, 0.1658764409071127, 0.20939885099737585], [-0.7052918250818047, 0.7252012632703241, -1.0133620699875483, 0.36606237518107415, 0.6449082576903957, -0.17473991818858178, 0.8754650404488951], [-0.5101360579238641, 0.41818395245977424, -0.7193869867649895, -0.3503899257051551, -0.11424538262980503, -0.2997110881627615, -0.1768348379366028], [0.3085766918048785, -0.17294828463558234, -0.19968853432988296, 0.01807645288703558, 0.28065221491892534, -0.5192204219887737, -0.026577569748623936]], [[-0.06543889329547663, 0.3630349207200558, -0.23780222403451642, 1.3274111406974334, 0.24415935337103978, 0.8787335034127547, -0.37989530562381885], [-0.14561185789074788, 0.23927895516539258, -0.8941808059574659, -0.7491783188769697, -0.16391887968668087, -0.43000605960384675, 0.37864075150634824], [0.00821081638745715, 0.5423781112739444, -0.04306047013847379, -0.7115559729759448, -0.3207456779392636, -0.06901008215578615, 0.5707083551298033], [-0.37857462864760166, -0.3346833660961983, 0.8526210627759425, 0.11884742316057048, 0.04662064317495303, 0.8157906486486116, 0.5172542084612325], [0.07917353295857328, 0.00013459520668199687, 0.22458063531759476, 0.6839597974271018, 0.3993568595017225, -0.5736846456153026, 0.0001244382558379144]], [[0.23260567829627152, 0.7120072597263155, 0.0748911841941861, 0.1390688764729498, 0.30279037186192737, -0.46344132665266163, 0.0576771570985332], [0.9676200824647863, -0.18178798807538188, -0.6556438655050226, 0.2388345366355657, 0.1345076548442881, -0.43335478190928844, -0.3516914654724372], [-0.12708735739735225, 0.22475394192380646, -0.07869826438975801, 0.15282683879210024, 0.6913634486307063, 0.7181215216685688, 0.10931114344899541], [-0.17898762496574386, 0.32869846681940695, 0.5879501046132042, -0.07475557395027937, 0.4814231917414812, -0.6940313821804548, -0.2032143968279263], [0.14144528849254986, 0.3604807786804876, -0.7794142695230756, -0.49117428496807497, -0.6048671998556744, 0.281704498499245, 0.5151449760238376]]], "dtype": "float64"}, {"__nd__": [[[-0.16075894247846817, 0.3125336862146319, -0.18076208649603304, 0.18192733934222724, -0.7883711038823631], [0.6577465557996967, 0.05191387417804236, -0.8198368220131113, 0.41562192322676417, -0.8423450872155513], [0.11336663575873106, -0.09808007567753155, 0.5153305599887175, -0.419092678693025, -0.44966945393834146], [-0.8640439340143662, -0.01379704456143444, -0.21464639509789965, -0.3952669226974095, -0.7022397882378614], [0.3403411538513887, -0.3130177556219595, 0.7535992682441495, -0.32730173497182197, 0.9558334786327962], [0.18830702421229684, -0.23385922811877638, -0.020832988009136806, 0.05604648585373701, 1.008696381345371], [-0.42815162251884104, -0.5877623607191236, -0.30318041295856357, 0.027036811209782755, 0.7024327596462128]], [[0.845202159901268, -1.2389734954949203, 0.7384380553929383, 0.8009107053141897, -0.8324570231663205], [0.269510100744109, 0.6977183004402401, 0.11283840901884598, 0.16797334846670295, -0.4228419045768107], [0.3366311813938711, 0.04702993543218851, -0.3138386781396554, 0.11395722356760511, 0.4332422916313579], [-0.6055647193657894, 0.335372413193081, -0.5281107476220982, -1.1015259193513667, 0.30120655475912295], [-0.24616257475378042, 1.0904938708654361, 0.7908350797094783, 0.9405485961516961, 0.39040865826463317], [-0.26272230912504535, -0.05598867182529021, 0.26914145923495947, 0.4134139943506048, 0.8244309919331515], [0.7151433836271744, -0.05207226361141182, 0.879082396553654, -0.6189110117559272, -0.5090759077307432]], [[0.23970285033249078, -0.07713053869924635, -0.06272159013439495, 0.45342234004339954, -0.5399978440567902], [0.5307935237899538, -0.26994344506568557, -0.3235802598200302, 0.939213235718408, 0.7067760572580646], [-0.4424143837538314, 0.7226286346261911, -0.03774645968838658, -0.6652495731816641, 0.12144331105251449], [-0.14422604755140256, 0.6128549614052725, 0.4880856025372115, 0.4419620154577947, -0.4402468433197321], [0.470751467972653, 0.37106340875630256, 0.622397578961162, 0.876273083020636, -0.1053057078915295], [-0.25197472262757675, -0.028394062826464252, -0.2736260936519378, 0.05202213881695466, 0.008360861234075855], [0.39212682587204956, 0.20870988247713493, 0.22108277622999958, 0.6558467440470533, 0.5988961231264394]], [[0.23662509100765697, -0.21743611307603353, -0.4829912335302463, -0.4897544409392944, -0.17728470880609803], [-0.15874150387319813, -0.14053783544702642, 0.1993749212436792, 0.1299142422031329, -0.17345160372192472], [-0.4462862629361029, -0.10441271681232753, 0.18278960771846448, -0.04368654697892672, 0.5079210502532089], [0.14973359665504604, 0.03011191569209091, -0.26168168187912594, 0.05095570053814634, 0.09241897699327643], [-0.5000489192525437, -0.8260524677202199, -0.16663564660551072, 0.7227154577859457, 0.746636037311921], [0.24525715176996052, 0.3109842351454637, 0.13797848912886304, -0.5397210605644853, -0.413822996626426], [0.6497470188945855, -0.3637862052365567, -0.20377279079199864, -0.6483232363103429, -0.358434512648307]], [[-0.04108652162343333, -0.2694911571842727, -0.16884370380796607, 0.2061802967540192, -0.6218392854852015], [0.39440493377991875, -0.44905948205895596, -0.16853616798876395, 0.7521012024241084, 0.33048662454923067], [-0.06660449320122398, 0.12725396219032842, -0.3660355342751908, -0.9516172392408028, 0.6331436716568465], [0.014778024497901079, -0.605371947940501, -0.5295346758852084, 0.10280271744374761, -0.050135175385959165], [0.6606454398749894, 0.3360102540296437, -0.0449132138899573, -0.006179222998979005, 0.5155348895767609], [-1.0318989235109781, 0.34752475324985355, 0.8110163787808162, -0.9819079450504836, -0.35689213175595463], [-0.12522588597860293, 0.26907974255771094, -0.9766704962688978, -0.06765776894546849, -0.06715562123014346]], [[0.24002468506846625, -0.37067382758115947, -0.18269884033055722, 0.059569857214784465, 0.2777421090010456], [-0.22694857240090735, 0.49272481457798784, -0.7367637454473434, -0.2006507045430574, 0.9013693133549181], [0.5974462069361848, -0.6090210942353659, 0.3710270948666453, -0.36845850006959147, -0.2507174506363178], [0.6019131426337748, -0.40611400454777574, -0.22146472962802366, 0.2638992116799055, 0.3096857041030245], [0.7297684181402286, -0.27303562947257026, -0.1153035539613704, 0.24722261638054116, 0.17591575107242355], [-0.927905325315962, -0.08918416425125095, 0.07183674796237027, -0.28813270361373144, 0.8844820274388052], [0.07424428906128211, 0.49188745766085207, 0.26214070895833186, -0.051746861219723835, -0.1489472419206125]]], "dtype": "float64"}, 3], "e": {"__tup__": [{"__nd__": [[0.08660265857858071, 0.3646714565984298, 0.2203993741571462, 0.5796775855491556, 0.13342169702157453], [0.22535701345796436, 0.1731089875209652, -0.07291352843452417, 0.33027677715040415, 0.37559868388842677], [0.40574892598521906, -0.5418155905987044, 0.13997655038467796, 0.7327025475518887, 0.4674563398647196], [-0.48394870973863513, -0.16008296151822432, -0.1802349759831053, -0.43463273011053283, 0.34628767758827517], [-0.039250618451997434, 0.016531906542975632, 0.02756739161008636, -0.059658548702402496, 0.04795399146394531], [0.34290132208968277, -0.008697150267361129, -0.054865551109814836, -0.06186259088377188, -0.7229779854323335], [-0.1800261560522873, 0.3470243648755225, 0.10363209942729389, 0.03238850247807794, 0.1590028054134346], [0.017216533950233336, -0.22599142747943315, -0.33884431699786965, -0.08524740086730748, -0.051005964531531126], [0.9746073658746915, -0.30253963950178436, -0.5008265095842979, -0.15486718844163794, 0.12873123323673513]], "dtype": "float64"}, {"__nd__": [[true, true, false], [true, false, true], [true, true, true], [true, true, false], [true, false, false], [true, false, false], [true, false, false], [true, true, true], [true, true, false]], "dtype": "bool"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(moe_forward, json.loads(_TESTS), 'torch', *[1e-06, 1e-08])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")